<a href="https://colab.research.google.com/github/AnthonyNorwood124/BrokeBull/blob/main/Stage_2_and_Options.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip -q install yfinance pandas numpy requests pyarrow


!pip -q install reportlab




import os
os.environ["PDF_ENABLED"] = "true"
os.environ["PDF_ATTACH_ONLY"] = "true"
os.environ["PDF_MAX_ROWS"] = "50"
os.environ["PDF_LOGO_PATH"] = ""  # optional; put a real path if you want branding


import os

os.environ["SMTP_USER"] = "anthonynorwood124@gmail.com"
os.environ["SMTP_PASS"] = "dhglbqtfviawevfi"
os.environ["EMAIL_TO"]  = ",".join(["smoke23664@gmail.com"])
os.environ["EMAIL_FROM"] = "BrokeBull Investments <anthonynorwood124@gmail.com>"
os.environ["EMAIL_ATTACH_TECH"] = "false"
os.environ["EMAIL_ATTACH_PRODUCT"] = "true"

import os
os.environ["SMTP_USER"]             = "anthonynorwood124@gmail.com"
os.environ["SMTP_PASS"]             = "dhglbqtfviawevfi"
os.environ["EMAIL_TO"]              = ",".join(["smoke23664@gmail.com"])
os.environ["EMAIL_FROM"]            = "BrokeBull Investments <anthonynorwood124@gmail.com>"

# Position sizing
os.environ["ACCOUNT_SIZE"]          = "10000"
os.environ["RISK_PER_TRADE_PCT"]    = "1.0"
os.environ["POSITION_STOP_ATR_MULT"]= "2.0"

# FMP fallback (optional but recommended)
os.environ["FMP_API_KEY"]           = ""   # get free key at financialmodelingprep.com

# Watchlist
os.environ["EMAIL_ATTACH_WATCHLIST"]= "true"

# Options scanner — relaxed for OTM-friendly spreads
os.environ["OPT_MAX_SPREAD_PCT"]    = "0.08"   # was 0.05 — OTM contracts have wider spreads
os.environ["EMAIL_ATTACH_OPTIONS"]  = "true"

# -*- coding: utf-8 -*-
"""
OCC / BrokeBull — Stage 2 Engine v2.3.3
================================================================================
v2.3.3 changes vs v2.3.2:
  · Added explicit attachment-build diagnostics (so we can see exactly which
    files are queued for email and why each one is/isn't included)

v2.3.2 changes vs v2.3.1:
  Funnel-driven relaxation (data-tuned from 321-TECH run):
    · MAX_PE:           35   → 50      (was rejecting growth premiums)
    · MAX_EV_EBITDA:    16   → 25      (47% failure rate at 16)
    · MIN_ROE_PCT:       8.0 → 5.0     (44% failure rate at 8)
    · MIN_FCF_YIELD_PCT: 2.0 → 0.0     (40% failure rate; reinvesting growth co's)
    · MAX_PEG:           2.0 → 2.5
    · MAX_PS:           10   → 12
  Sector-relative multipliers loosened from 1.20-1.25 to 1.35-1.40 across the board.

v2.3.1 changes vs v2.3:
  · MAX_DEBT_TO_EQUITY: 1.5 → 200.0  (yfinance returns D/E as percent, not ratio)
  · MIN_INTEREST_COVERAGE: 3.0 → 2.0
  · OPT_MAX_SPREAD_PCT: 0.05 → 0.08  (OTM contracts have wider spreads)
  · Added funnel-failure diagnostic logs before QUALITY filter

v2.3 changes vs v2.2:
  OPTIONS LAYER — NEW
    Standalone options scanner for a fixed high-liquidity watchlist.
    Runs independently of the Stage 2 pipeline.
      · Watchlist: SPY, QQQ, IWM + Mag 7 (AAPL MSFT NVDA TSLA AMZN GOOG META)
      · Sides:     Calls AND Puts (full flexibility)
      · Moneyness: ATM/OTM only (no ITM)
      · Delta:     0.30-0.50 band for calls, -0.50 to -0.30 for puts (target 0.40)
      · DTE:       30-90 days
      · Output:    stage2_OPTIONS_<date>.csv  (attached to daily email)

  All v2.2 features retained:
    UNIVERSE — 12 sources (NASDAQ/Other + S&P 500/400/600 + Russell 1000/2000/3000
               + DJ Wilshire 5000 + MSCI World/EAFE + Nikkei 225)
    REC #1 Parquet cache · REC #2 Parallel scoring · REC #3 Delta tracking
    REC #4 Retry/backoff · REC #5 Extra columns · REC #6 Dedup weekly resample
    UPGRADE 1-4 (sector email / watchlist / position sizing / FMP fallback)

================================================================================
"""

import os
import io
import re
import math
import ssl
import json
import time
import smtplib
import datetime as dt
import threading
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import requests
import yfinance as yf
from email.message import EmailMessage
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from concurrent.futures import ThreadPoolExecutor, as_completed

import logging
logging.getLogger("yfinance").setLevel(logging.CRITICAL)
logging.getLogger("urllib3").setLevel(logging.CRITICAL)


# ══════════════════════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════════════════════

@dataclass(frozen=True)
class CFG:
    # Price history
    PERIOD_STR: str = "10y"
    YF_CHUNK: int = 200
    YF_THREADS: bool = True

    # Stage-2 gate (WEEKLY chart — 40W ≈ daily 200 SMA)
    STAGE2_WEEKLY_MA_WEEKS: int = int(os.getenv("STAGE2_WEEKLY_MA_WEEKS", "40"))
    STAGE2_WEEKLY_SLOPE_LOOKBACK_WK: int = int(os.getenv("STAGE2_WEEKLY_SLOPE_LOOKBACK_WK", "20"))

    # Purist daily 200-SMA confirmation (optional second gate)
    DAILY_SMA_DAYS: int = int(os.getenv("DAILY_SMA_DAYS", "200"))
    DAILY_SLOPE_LOOKBACK_DAYS: int = int(os.getenv("DAILY_SLOPE_LOOKBACK_DAYS", "20"))
    REQUIRE_DAILY_200SMA_CONFIRM: bool = os.getenv("REQUIRE_DAILY_200SMA_CONFIRM", "false").strip().lower() == "true"

    # Liquidity
    MIN_PRICE: float = float(os.getenv("MIN_PRICE", "1.0"))
    MIN_AVG_VOL_30D: float = float(os.getenv("MIN_AVG_VOL_30D", "100000"))

    # RS vs SPY (weekly)
    RS_MA_WEEKS: int = int(os.getenv("RS_MA_WEEKS", "30"))
    RS_SLOPE_LOOKBACK_WK: int = int(os.getenv("RS_SLOPE_LOOKBACK_WK", "13"))
    RS_REQUIRE_ABOVE_MA: bool = os.getenv("RS_REQUIRE_ABOVE_MA", "false").strip().lower() == "true"
    RS_REQUIRE_RISING: bool = os.getenv("RS_REQUIRE_RISING", "true").strip().lower() == "true"

    # Base breakout
    BASE_MIN_WEEKS: int = int(os.getenv("BASE_MIN_WEEKS", "6"))
    BASE_MAX_WEEKS: int = int(os.getenv("BASE_MAX_WEEKS", "30"))
    BASE_MAX_DEPTH: float = float(os.getenv("BASE_MAX_DEPTH", "0.35"))
    BREAKOUT_LOOKBACK_DAYS: int = int(os.getenv("BREAKOUT_LOOKBACK_DAYS", "10"))
    BREAKOUT_MIN_VOL_MULT: float = float(os.getenv("BREAKOUT_MIN_VOL_MULT", "1.0"))
    BREAKOUT_BUFFER_PCT: float = float(os.getenv("BREAKOUT_BUFFER_PCT", "0.003"))
    RETEST_FLOOR_PCT: float = float(os.getenv("RETEST_FLOOR_PCT", "0.95"))

    # Trend template
    TREND_MA_FAST: int = 50
    TREND_MA_MID: int = 150
    TREND_MA_SLOW: int = 200
    TREND_52W_DAYS: int = 252
    TREND_OK_SCORE: float = float(os.getenv("TREND_OK_SCORE", "60"))

    # Sector-relative valuation filters
    # v2.3.2: relaxed multipliers (was 1.20-1.25, now 1.35-1.40) to keep growth
    # premiums within sectors while maintaining outlier discipline.
    RELATIVE_FILTER_ENABLED: bool = os.getenv("RELATIVE_FILTER_ENABLED", "true").strip().lower() == "true"
    REL_THRESHOLDS = {
        "PE_MAX_MULT":        1.40,
        "EV_EBITDA_MAX_MULT": 1.40,
        "PEG_MAX_MULT":       1.35,
        "PS_MAX_MULT":        1.40,
        "PB_MAX_MULT":        1.40,
        "NET_MARGIN_MIN_PC":  -1.0,
        "ROE_MIN_PC":         -1.0,
        "FCF_YIELD_MIN_PC":   -1.0,
    }

    # Absolute valuation
    # NOTE (v2.3.2): Thresholds tuned from funnel diagnostics on 321-pass run.
    # Previous v2.3 / v2.3.1 thresholds (commented) were rejecting healthy growth
    # names. Major relaxations: PE 35→50, EV/EBITDA 16→25, ROE 8→5,
    # FCF yield 2→0. D/E remains at 200 (yfinance percent units, not ratio).
    STRICT_VALUATION: bool = os.getenv("STRICT_VALUATION", "false").strip().lower() == "true"
    VAL_THRESHOLDS = {
        "MIN_EPS_TTM":            0.0,    # unchanged — keep profitability gate
        "MIN_ROE_PCT":            5.0,    # was 8.0 — too aggressive (43.9% failed)
        "MIN_NET_MARGIN_PCT":     0.0,    # unchanged — profitability gate
        "MAX_PE":                50.0,    # was 35.0 — allow growth premiums
        "MAX_PEG":                2.5,    # was 2.0 — slightly relaxed
        "MAX_PB":                10.0,    # unchanged — already permissive
        "MAX_PS":                12.0,    # was 10.0 — slight relaxation
        "MAX_EV_EBITDA":         25.0,    # was 16.0 — biggest offender (47.4% failed)
        "MIN_FCF_YIELD_PCT":      0.0,    # was 2.0 — growth companies reinvest aggressively
        "MAX_DEBT_TO_EQUITY":   200.0,    # yfinance returns as % not ratio
        "MIN_INTEREST_COVERAGE":  2.0,
    }

    # Parallelism (REC #2)
    SCORE_WORKERS: int = int(os.getenv("SCORE_WORKERS", "16"))
    FUND_MAX_WORKERS: int = int(os.getenv("FUND_MAX_WORKERS", "6"))

    # Universe — extended index fetching (v2.1+)
    UNIVERSE_WORKERS: int = int(os.getenv("UNIVERSE_WORKERS", "4"))
    UNIVERSE_INCLUDE_SP: bool = os.getenv("UNIVERSE_INCLUDE_SP", "true").strip().lower() == "true"
    UNIVERSE_INCLUDE_RUSSELL: bool = os.getenv("UNIVERSE_INCLUDE_RUSSELL", "true").strip().lower() == "true"
    UNIVERSE_INCLUDE_WILSHIRE: bool = os.getenv("UNIVERSE_INCLUDE_WILSHIRE", "true").strip().lower() == "true"
    UNIVERSE_INCLUDE_MSCI: bool = os.getenv("UNIVERSE_INCLUDE_MSCI", "true").strip().lower() == "true"
    UNIVERSE_INCLUDE_NIKKEI: bool = os.getenv("UNIVERSE_INCLUDE_NIKKEI", "true").strip().lower() == "true"

    # Output
    OUT_DIR: str = os.getenv("OUT_DIR", "./output")
    CACHE_DIR: str = os.getenv("CACHE_DIR", "./cache")

    # Cache (REC #1)
    FORCE_PRICE_REFRESH: bool = os.getenv("FORCE_PRICE_REFRESH", "false").strip().lower() == "true"
    CACHE_KEEP_DAYS: int = int(os.getenv("CACHE_KEEP_DAYS", "7"))

    # Delta tracking (REC #3)
    DELTA_ENABLED: bool = os.getenv("DELTA_ENABLED", "true").strip().lower() == "true"

    # OCC PRODUCT tier floor (REC #3 quality floor)
    PRODUCT_MIN_TIER: str = os.getenv("PRODUCT_MIN_TIER", "TIER_3_WATCH")
    PRODUCT_MAX_ROWS: int = int(os.getenv("PRODUCT_MAX_ROWS", "50"))

    # UPGRADE 1 — Sector breakdown in email
    EMAIL_SECTOR_BREAKDOWN: bool = os.getenv("EMAIL_SECTOR_BREAKDOWN", "true").strip().lower() == "true"

    # UPGRADE 2 — Watchlist (TECH pass, QUALITY fail candidates)
    WATCHLIST_ENABLED: bool = os.getenv("WATCHLIST_ENABLED", "true").strip().lower() == "true"
    WATCHLIST_FILENAME: str = os.getenv("WATCHLIST_FILENAME", "").strip()
    EMAIL_ATTACH_WATCHLIST: bool = os.getenv("EMAIL_ATTACH_WATCHLIST", "true").strip().lower() == "true"
    WATCHLIST_MAX_ROWS: int = int(os.getenv("WATCHLIST_MAX_ROWS", "100"))

    # UPGRADE 3 — Position sizing
    ACCOUNT_SIZE: float = float(os.getenv("ACCOUNT_SIZE", "100000"))
    RISK_PER_TRADE_PCT: float = float(os.getenv("RISK_PER_TRADE_PCT", "1.0"))
    POSITION_STOP_ATR_MULT: float = float(os.getenv("POSITION_STOP_ATR_MULT", "2.0"))

    # UPGRADE 4 — FMP fallback data source
    FMP_API_KEY: str = os.getenv("FMP_API_KEY", "").strip()
    FMP_ENABLED: bool = os.getenv("FMP_ENABLED", "true").strip().lower() == "true"
    FMP_TIMEOUT: int = int(os.getenv("FMP_TIMEOUT", "10"))

    # OPTIONS LAYER (v2.3) — attach options CSV to email
    EMAIL_ATTACH_OPTIONS: bool = os.getenv("EMAIL_ATTACH_OPTIONS", "true").strip().lower() == "true"

    # Email
    SMTP_HOST: str = os.getenv("SMTP_HOST", "smtp.gmail.com")
    SMTP_PORT: int = int(os.getenv("SMTP_PORT", "587"))
    SMTP_USER: str = os.getenv("SMTP_USER", "").strip()
    SMTP_PASS: str = os.getenv("SMTP_PASS", "").strip()
    SMTP_TLS: bool = os.getenv("SMTP_TLS", "true").strip().lower() == "true"
    EMAIL_FROM: str = os.getenv("EMAIL_FROM", "").strip()
    EMAIL_TO: str = os.getenv("EMAIL_TO", "").strip()

    EMAIL_ATTACH_TECH: bool = os.getenv("EMAIL_ATTACH_TECH", "false").lower().strip() == "true"
    EMAIL_ATTACH_PRODUCT: bool = os.getenv("EMAIL_ATTACH_PRODUCT", "true").lower().strip() == "true"
    PRODUCT_FILENAME: str = os.getenv("PRODUCT_FILENAME", "").strip()


# ══════════════════════════════════════════════════════════════════════════════
# OPTIONS LAYER  (v2.3 — scans SPY/QQQ/IWM + Mag 7, calls & puts, OTM/ATM only)
# ══════════════════════════════════════════════════════════════════════════════

OPTIONS_WATCHLIST: List[str] = [
    # Index ETFs
    "SPY", "QQQ", "IWM",
    # Mag 7
    "AAPL", "MSFT", "NVDA", "TSLA", "AMZN", "GOOG", "META",
]

# Options config — overridable via env vars
OPT_MIN_DTE                = int(os.getenv("OPT_MIN_DTE", "30"))
OPT_MAX_DTE                = int(os.getenv("OPT_MAX_DTE", "90"))
OPT_MIN_VOLUME             = int(os.getenv("OPT_MIN_VOLUME", "500"))
OPT_MIN_OI                 = int(os.getenv("OPT_MIN_OI", "1000"))
OPT_MAX_SPREAD_PCT         = float(os.getenv("OPT_MAX_SPREAD_PCT", "0.05"))
# Delta targets tuned for OTM/ATM focus (no ITM). Calls sit below 0.50 delta.
OPT_TARGET_DELTA_CALL      = float(os.getenv("OPT_TARGET_DELTA_CALL", "0.40"))
OPT_TARGET_DELTA_PUT       = float(os.getenv("OPT_TARGET_DELTA_PUT", "-0.40"))
OPT_TARGET_DELTA_LOW       = float(os.getenv("OPT_TARGET_DELTA_LOW", "0.30"))
OPT_TARGET_DELTA_HIGH      = float(os.getenv("OPT_TARGET_DELTA_HIGH", "0.50"))
OPT_MAX_PER_UNDERLYING     = int(os.getenv("OPT_MAX_PER_UNDERLYING", "3"))
OPT_TOP_N                  = int(os.getenv("OPT_TOP_N", "50"))

# Scoring weights (sum to 1.0)
OPT_W_DELTA_FIT = 0.45
OPT_W_LIQUIDITY = 0.45
OPT_W_PREMIUM   = 0.10

OPT_FETCH_RETRIES = 2
OPT_FETCH_SLEEP   = 0.4


def _opt_get_underlying_price(ticker: str) -> Optional[float]:
    """Fetch latest close for the underlying. Returns None on failure."""
    try:
        hist = yf.Ticker(ticker).history(period="5d", interval="1d", auto_adjust=True)
        if hist is None or hist.empty:
            return None
        close = float(hist["Close"].iloc[-1])
        return close if (math.isfinite(close) and close > 0) else None
    except Exception:
        return None


def _opt_estimate_moneyness(side: str, underlying_price: float, strike: float) -> str:
    if underlying_price <= 0 or not np.isfinite(strike):
        return "UNK"
    if side == "call":
        if strike < underlying_price * 0.98:
            return "ITM"
        if strike <= underlying_price * 1.02:
            return "ATM"
        return "OTM"
    # put
    if strike > underlying_price * 1.02:
        return "ITM"
    if strike >= underlying_price * 0.98:
        return "ATM"
    return "OTM"


def _opt_normalize_series(s: pd.Series) -> pd.Series:
    """Min-max normalize to [0,1]. Constant series -> 0.5."""
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.min(skipna=True), s.max(skipna=True)
    if pd.isna(lo) or pd.isna(hi) or hi == lo:
        return pd.Series(0.5, index=s.index)
    return (s - lo) / (hi - lo)


def _opt_fetch_chain_with_retry(tk: yf.Ticker, exp: str):
    """yfinance option_chain with linear backoff."""
    last_err = None
    for attempt in range(OPT_FETCH_RETRIES + 1):
        try:
            return tk.option_chain(exp)
        except Exception as e:
            last_err = e
            time.sleep(OPT_FETCH_SLEEP * (attempt + 1))
    raise last_err if last_err else RuntimeError("option_chain failed")


def _opt_rank_contracts(chain_df: pd.DataFrame, underlying_price: float,
                        expiration: str, side: str) -> pd.DataFrame:
    """Score a single expiration's chain (calls OR puts), filter, and rank."""
    if chain_df is None or chain_df.empty:
        return pd.DataFrame()

    x = chain_df.copy()
    numeric_cols = ["bid", "ask", "lastPrice", "volume", "openInterest",
                    "impliedVolatility", "strike", "delta"]
    for col in numeric_cols:
        if col in x.columns:
            x[col] = pd.to_numeric(x[col], errors="coerce")
        else:
            x[col] = np.nan

    # Only compute mid when both sides of the quote exist
    x["mid"] = np.where(
        (x["bid"] > 0) & (x["ask"] > 0),
        (x["bid"] + x["ask"]) / 2.0,
        np.nan,
    )
    x["spread_pct"] = np.where(
        x["mid"].notna() & (x["mid"] > 0),
        (x["ask"] - x["bid"]) / x["mid"],
        np.nan,
    )
    x["moneyness"] = x["strike"].apply(
        lambda s: _opt_estimate_moneyness(side, underlying_price, s)
    )
    x["side"] = side

    # Delta fit: target +0.40 for calls, -0.40 for puts (both OTM).
    target_delta = OPT_TARGET_DELTA_CALL if side == "call" else OPT_TARGET_DELTA_PUT
    moneyness_fallback = x["moneyness"].map(
        {"OTM": 0.85, "ATM": 0.75, "ITM": 0.10, "UNK": 0.0}
    ).fillna(0.0)
    delta_fit_from_delta = 1.0 - np.minimum(np.abs(x["delta"] - target_delta), 1.0)
    x["delta_fit_score"] = np.where(
        x["delta"].notna(), delta_fit_from_delta, moneyness_fallback
    )
    x["delta_fit_score"] = x["delta_fit_score"].clip(0, 1)

    # Liquidity: log-scale vol + OI, spread tightness. Normalized to [0,1].
    vol_component = np.log1p(x["volume"].fillna(0))
    oi_component = np.log1p(x["openInterest"].fillna(0))
    spread_tightness = 1 - x["spread_pct"].clip(lower=0, upper=1).fillna(1)
    raw_liquidity = vol_component * 0.45 + oi_component * 0.45 + spread_tightness * 0.10
    x["liquidity_score"] = _opt_normalize_series(raw_liquidity)

    # Premium: cheaper mid slightly preferred as tiebreaker. Normalized.
    inv_mid = np.where(x["mid"] > 0, 1.0 / np.log1p(x["mid"]), 0.0)
    x["premium_score"] = _opt_normalize_series(pd.Series(inv_mid, index=x.index))

    x["final_score"] = (
        x["delta_fit_score"] * OPT_W_DELTA_FIT
        + x["liquidity_score"] * OPT_W_LIQUIDITY
        + x["premium_score"] * OPT_W_PREMIUM
    )
    x["expiration"] = expiration

    # Hard filters: liquidity + ATM/OTM only (no ITM)
    filters = (
        (x["volume"].fillna(0) >= OPT_MIN_VOLUME)
        & (x["openInterest"].fillna(0) >= OPT_MIN_OI)
        & (x["spread_pct"].notna())
        & (x["spread_pct"] <= OPT_MAX_SPREAD_PCT)
        & (x["moneyness"].isin(["ATM", "OTM"]))
    )

    # Delta band (when provider supplies delta).
    # Calls: 0.30-0.50 (OTM zone). Puts: -0.50 to -0.30 (same band, negated).
    if x["delta"].notna().any():
        if side == "call":
            in_band = x["delta"].between(OPT_TARGET_DELTA_LOW, OPT_TARGET_DELTA_HIGH)
        else:
            in_band = x["delta"].between(-OPT_TARGET_DELTA_HIGH, -OPT_TARGET_DELTA_LOW)
        filters &= (in_band | x["delta"].isna())

    x = x.loc[filters].copy()
    if x.empty:
        return x

    keep_cols = [
        "contractSymbol", "side", "strike", "bid", "ask", "mid", "lastPrice",
        "volume", "openInterest", "impliedVolatility", "delta", "moneyness",
        "spread_pct", "expiration", "final_score",
    ]
    keep_cols = [c for c in keep_cols if c in x.columns]
    return x[keep_cols].sort_values("final_score", ascending=False)


def _opt_get_best_contracts_for_ticker(ticker: str, underlying_price: float) -> pd.DataFrame:
    """Scan all eligible expirations for a ticker and return top calls + puts."""
    tk = yf.Ticker(ticker)

    try:
        expirations = tk.options or []
    except Exception:
        return pd.DataFrame()
    if not expirations:
        return pd.DataFrame()

    today = dt.date.today()
    picked_frames: List[pd.DataFrame] = []

    for exp in expirations:
        try:
            exp_date = dt.datetime.strptime(exp, "%Y-%m-%d").date()
        except Exception:
            continue

        dte = (exp_date - today).days
        if dte < OPT_MIN_DTE or dte > OPT_MAX_DTE:
            continue

        try:
            chain = _opt_fetch_chain_with_retry(tk, exp)
        except Exception:
            continue
        if chain is None:
            continue

        # Rank calls and puts separately
        for side, df in (("call", getattr(chain, "calls", None)),
                         ("put",  getattr(chain, "puts",  None))):
            if df is None or df.empty:
                continue
            ranked = _opt_rank_contracts(
                df.copy(), underlying_price=underlying_price,
                expiration=exp, side=side,
            )
            if ranked.empty:
                continue
            ranked["ticker"] = ticker
            ranked["underlying_price"] = underlying_price
            ranked["dte"] = dte
            picked_frames.append(ranked.head(OPT_MAX_PER_UNDERLYING))

    if not picked_frames:
        return pd.DataFrame()

    out = pd.concat(picked_frames, ignore_index=True)
    # Keep top N per (ticker, side) so calls & puts both represented
    out = out.sort_values(
        ["ticker", "side", "final_score"], ascending=[True, True, False]
    )
    return (
        out.groupby(["ticker", "side"], as_index=False, group_keys=False)
           .head(OPT_MAX_PER_UNDERLYING)
    )


def run_options_scan(out_dir: str, today: Optional[str] = None) -> str:
    """
    Scan the fixed options watchlist (SPY/QQQ/IWM + Mag 7) and write results.
    Returns output path (empty string if nothing written).
    """
    if today is None:
        today = dt.date.today().isoformat()
    os.makedirs(out_dir, exist_ok=True)

    print(f"\n[Options] Scanning {len(OPTIONS_WATCHLIST)} underlyings: "
          f"{', '.join(OPTIONS_WATCHLIST)}")

    all_frames: List[pd.DataFrame] = []
    for ticker in OPTIONS_WATCHLIST:
        underlying_price = _opt_get_underlying_price(ticker)
        if underlying_price is None:
            print(f"[Options] {ticker}: skipped (no underlying price)")
            continue

        try:
            contracts = _opt_get_best_contracts_for_ticker(ticker, underlying_price)
        except Exception as e:
            print(f"[Options] {ticker}: error — {e}")
            continue

        if contracts.empty:
            print(f"[Options] {ticker}: no qualifying contracts "
                  f"(underlying=${underlying_price:.2f})")
            continue

        n_calls = (contracts["side"] == "call").sum()
        n_puts  = (contracts["side"] == "put").sum()
        print(f"[Options] {ticker}: underlying=${underlying_price:.2f} | "
              f"{n_calls} calls, {n_puts} puts kept")
        all_frames.append(contracts)

    if not all_frames:
        print("[Options] No qualifying contracts across watchlist.")
        return ""

    final = pd.concat(all_frames, ignore_index=True)
    final = final.sort_values("final_score", ascending=False).head(OPT_TOP_N)

    order = [
        "ticker", "side", "underlying_price", "contractSymbol", "strike",
        "expiration", "dte", "moneyness", "bid", "ask", "mid", "lastPrice",
        "volume", "openInterest", "impliedVolatility", "delta", "spread_pct",
        "final_score",
    ]
    final = final.reindex(columns=[c for c in order if c in final.columns])

    out_path = os.path.join(out_dir, f"stage2_OPTIONS_{today}.csv")
    final.to_csv(out_path, index=False)
    print(f"[Options] Wrote {out_path} ({len(final)} contracts)")
    return out_path


# ══════════════════════════════════════════════════════════════════════════════
# REC #1 — PARQUET PRICE CACHE
# ══════════════════════════════════════════════════════════════════════════════

def _cache_paths(cache_dir: str, period: str, today: str) -> Tuple[str, str]:
    base = os.path.join(cache_dir, f"prices_{today}_{period}")
    return base + ".parquet", base + ".meta.json"


def _load_cache(parquet_path: str, meta_path: str) -> Tuple[Optional[pd.DataFrame], Optional[dict]]:
    if not (os.path.exists(parquet_path) and os.path.exists(meta_path)):
        return None, None
    try:
        df = pd.read_parquet(parquet_path)
        with open(meta_path) as f:
            meta = json.load(f)
        return df, meta
    except Exception as e:
        print(f"[Cache] Load error ({e}) — will re-download.")
        return None, None


def _save_cache(long_df: pd.DataFrame, meta: dict, parquet_path: str, meta_path: str) -> None:
    os.makedirs(os.path.dirname(parquet_path) or ".", exist_ok=True)
    long_df.to_parquet(parquet_path, index=False, compression="snappy", engine="pyarrow")
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)
    size_mb = os.path.getsize(parquet_path) / 1e6
    print(f"[Cache] Saved → {parquet_path} ({size_mb:.1f} MB, {meta['ticker_count']} tickers)")


def _wide_to_dict(long_df: pd.DataFrame) -> Dict[str, pd.DataFrame]:
    out: Dict[str, pd.DataFrame] = {}
    for ticker, grp in long_df.groupby("ticker"):
        out[str(ticker)] = grp.drop(columns=["ticker"]).sort_values("date").reset_index(drop=True)
    return out


def _dict_to_long(prices: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    frames = [df.assign(ticker=t) for t, df in prices.items()]
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def purge_old_cache(cache_dir: str, keep_days: int = 7) -> List[str]:
    deleted: List[str] = []
    cutoff = dt.date.today() - dt.timedelta(days=keep_days)
    if not os.path.exists(cache_dir):
        return deleted
    for fname in os.listdir(cache_dir):
        parts = fname.split("_")
        if len(parts) < 2:
            continue
        try:
            file_date = dt.date.fromisoformat(parts[1])
        except ValueError:
            continue
        if file_date < cutoff:
            fpath = os.path.join(cache_dir, fname)
            os.remove(fpath)
            deleted.append(fpath)
            print(f"[Cache] Purged: {fname}")
    return deleted


def cache_info(cache_dir: str) -> pd.DataFrame:
    rows = []
    if not os.path.exists(cache_dir):
        return pd.DataFrame()
    for fname in sorted(os.listdir(cache_dir)):
        if not fname.endswith(".meta.json"):
            continue
        meta_path = os.path.join(cache_dir, fname)
        parquet_path = meta_path.replace(".meta.json", ".parquet")
        try:
            with open(meta_path) as f:
                meta = json.load(f)
            size_mb = os.path.getsize(parquet_path) / 1e6 if os.path.exists(parquet_path) else 0.0
            rows.append({
                "date": meta.get("date"),
                "period": meta.get("period"),
                "tickers": meta.get("ticker_count"),
                "fetched_at": meta.get("fetch_timestamp"),
                "bad_symbols": len(meta.get("bad_symbols", [])),
                "size_mb": round(size_mb, 2),
            })
        except Exception:
            continue
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# EMAIL  — robust, self-diagnosing, HTML body
# ══════════════════════════════════════════════════════════════════════════════

def _parse_recipients(to_field: str) -> List[str]:
    """Return a clean list of recipient addresses from a comma/semicolon string."""
    if not to_field:
        return []
    return [p.strip() for p in re.split(r"[;,]+", str(to_field)) if p.strip()]


def diagnose_email_config(cfg: CFG) -> Tuple[bool, List[str]]:
    issues = []
    smtp_user = os.getenv("SMTP_USER", "").strip()
    smtp_pass = os.getenv("SMTP_PASS", "").strip()
    email_to  = os.getenv("EMAIL_TO",  "").strip()
    email_from= os.getenv("EMAIL_FROM","").strip()

    if not smtp_user:  issues.append("SMTP_USER not set  (your Gmail address)")
    if not smtp_pass:  issues.append("SMTP_PASS not set  (Gmail App Password — 16 chars, no spaces)")
    if not email_to:   issues.append("EMAIL_TO not set   (recipient address, comma-separated for multiple)")
    if not email_from: issues.append("EMAIL_FROM not set (optional — defaults to SMTP_USER if blank)")

    print("\n┌─ Email Pre-flight ──────────────────────────────────────────")
    print(f"│  SMTP_USER  : {'✓ ' + smtp_user if smtp_user else '✗  NOT SET'}")
    print(f"│  SMTP_PASS  : {'✓  [set]'       if smtp_pass else '✗  NOT SET'}")
    print(f"│  EMAIL_FROM : {'✓ ' + email_from if email_from else '○  (will use SMTP_USER)'}")
    print(f"│  EMAIL_TO   : {'✓ ' + email_to   if email_to  else '✗  NOT SET'}")
    print(f"│  SMTP_HOST  : {os.getenv('SMTP_HOST','smtp.gmail.com')}:{os.getenv('SMTP_PORT','587')}")
    print(f"│  Status     : {'READY' if not issues else 'NOT READY — see issues below'}")
    for issue in issues:
        print(f"│  ⚠  {issue}")
    print("└────────────────────────────────────────────────────────────\n")

    return (len(issues) == 0), issues


def _build_html_body(today: str, tech_df: pd.DataFrame, quality_df: pd.DataFrame,
                     product_df: pd.DataFrame, cfg: CFG,
                     watchlist_df=None, sector_data: Optional[dict] = None) -> str:
    """UPGRADE 1 — Rich HTML email: sector breakdown, watchlist, position sizing columns."""
    top = product_df.head(15).copy() if not product_df.empty else pd.DataFrame()

    def row_color(tier):
        return {"TIER_1_CORE":"#1a472a","TIER_2_HIGH":"#1e3a5f",
                "TIER_3_WATCH":"#3b2f0a"}.get(str(tier), "#1a1a2e")

    def badge(val):
        c = {"STRONG BUY":"#00c853","BUY":"#2979ff","WATCH":"#ffd600","AVOID":"#d50000",
             "new":"#00bcd4","tier_upgrade":"#76ff03","tier_downgrade":"#ff6d00",
             "unchanged":"#555"}.get(str(val), "#555")
        return f'<span style="background:{c};color:#000;padding:2px 7px;border-radius:4px;font-size:11px;font-weight:bold">{val}</span>'

    def fmt(v, prefix="", suffix="", decimals=1):
        try:
            f = float(v)
            if not math.isfinite(f): return "\u2014"
            return f"{prefix}{f:,.{decimals}f}{suffix}"
        except Exception:
            return str(v) if v not in (None, "", float("nan")) else "\u2014"

    # ── Product rows ──────────────────────────────────────────────────────────
    prod_rows = ""
    if not top.empty:
        for _, r in top.iterrows():
            bg = row_color(str(r.get("occ_tier","")))
            prod_rows += (
                f'<tr style="background:{bg};color:#eee;">' +
                f'<td style="padding:6px 10px;font-weight:bold;color:#fff">{r.get("ticker","")}</td>' +
                f'<td>{badge(r.get("signal",""))}</td>' +
                f'<td style="text-align:center">{fmt(r.get("occ_score"))}</td>' +
                f'<td style="text-align:center">{fmt(r.get("close"),"$",decimals=2)}</td>' +
                f'<td style="text-align:center">{fmt(r.get("atr_pct"),suffix="%")}</td>' +
                f'<td style="text-align:center">{fmt(r.get("rel_volume"))}</td>' +
                f'<td style="text-align:center">{fmt(r.get("days_since_breakout"),decimals=0)}</td>' +
                f'<td>{badge(r.get("signal_change",""))}</td>' +
                f'<td style="text-align:center">{fmt(r.get("days_in_product"),decimals=0)}</td>' +
                f'<td style="text-align:center;font-size:11px">{r.get("earnings_date","\u2014")}</td>' +
                f'<td style="text-align:center;color:#4fc3f7">{fmt(r.get("pos_dollars"),"$",decimals=0)}</td>' +
                f'<td style="text-align:center;color:#4fc3f7">{fmt(r.get("pos_pct_account"),suffix="%")}</td>' +
                f'<td style="text-align:center;color:#ff6b6b">{fmt(r.get("stop_price"),"$",decimals=2)}</td>' +
                f'<td style="font-size:11px;color:#aaa">{r.get("sector","")}</td>' +
                "</tr>"
            )
    else:
        prod_rows = '<tr><td colspan="14" style="text-align:center;color:#aaa;padding:20px">No picks passed filters today</td></tr>'

    # ── UPGRADE 1: Sector breakdown ───────────────────────────────────────────
    sector_html = ""
    if cfg.EMAIL_SECTOR_BREAKDOWN and sector_data and sector_data.get("product"):
        sec_colors = {
            "Technology":"#1565c0","Healthcare":"#1b5e20","Industrials":"#4a148c",
            "Energy":"#e65100","Consumer Cyclical":"#880e4f","Financial Services":"#01579b",
            "Communication Services":"#006064","Consumer Defensive":"#33691e",
            "Basic Materials":"#3e2723","Utilities":"#37474f","Real Estate":"#4e342e",
        }
        sec_rows = ""
        for row in sector_data["product"]:
            sec  = str(row.get("sector","Other") or "Other")
            cnt  = row.get("count", 0)
            avg  = row.get("avg_occ", 0)
            tix  = row.get("tickers","")
            sc   = sec_colors.get(sec,"#1a1a2e")
            bw   = min(int(cnt * 20), 200)
            sec_rows += (
                '<tr style="background:#0d0d1a">' +
                f'<td style="padding:5px 10px"><span style="background:{sc};color:#fff;padding:2px 8px;border-radius:3px;font-size:11px">{sec}</span></td>' +
                f'<td style="text-align:center;font-weight:bold;color:#4fc3f7">{cnt}</td>' +
                f'<td style="text-align:center;color:#aaa">{avg}</td>' +
                f'<td style="padding:4px 10px"><div style="background:{sc};height:10px;width:{bw}px;border-radius:3px;opacity:.7"></div></td>' +
                f'<td style="font-size:11px;color:#888">{tix}</td>' +
                "</tr>"
            )
        sector_html = (
            '<h3 style="color:#4fc3f7;margin:24px 0 8px;font-size:14px">\U0001f4ca SECTOR CONCENTRATION</h3>' +
            '<table style="width:100%;border-collapse:collapse;font-size:12px;margin-bottom:20px">' +
            '<thead><tr style="background:#0d2137">' +
            '<th style="padding:7px 10px;color:#4fc3f7;text-align:left">Sector</th>' +
            '<th style="color:#4fc3f7">Count</th><th style="color:#4fc3f7">Avg OCC</th>' +
            '<th style="color:#4fc3f7;padding-left:10px">Weight</th>' +
            '<th style="color:#4fc3f7;text-align:left">Tickers</th>' +
            f"</tr></thead><tbody>{sec_rows}</tbody></table>"
        )

    # ── UPGRADE 2: Watchlist summary ──────────────────────────────────────────
    watch_html = ""
    if watchlist_df is not None and not watchlist_df.empty:
        w_rows = ""
        for _, r in watchlist_df.head(10).iterrows():
            w_rows += (
                '<tr style="background:#1a1a1a;color:#ccc">' +
                f'<td style="padding:5px 10px;font-weight:bold;color:#fff">{r.get("ticker","")}</td>' +
                f'<td style="text-align:center;color:#ffd600">{fmt(r.get("occ_score"))}</td>' +
                f'<td style="text-align:center">{fmt(r.get("close"),"$",decimals=2)}</td>' +
                f'<td style="font-size:11px;color:#ff6b6b">{r.get("failed_checks","")}</td>' +
                f'<td style="font-size:11px;color:#aaa">{r.get("sector","")}</td>' +
                f'<td style="text-align:center;font-size:11px">{r.get("earnings_date","\u2014")}</td>' +
                "</tr>"
            )
        wl_count = len(watchlist_df)
        watch_html = (
            f'<h3 style="color:#ffd600;margin:24px 0 8px;font-size:14px">\U0001f440 WATCHLIST \u2014 {wl_count} CANDIDATES (Tech Pass / Quality Fail)</h3>' +
            '<p style="color:#888;font-size:11px;margin:0 0 8px">Technically strong but failed valuation. Monitor for fundamental improvement.</p>' +
            '<table style="width:100%;border-collapse:collapse;font-size:12px;margin-bottom:20px">' +
            '<thead><tr style="background:#0d2137">' +
            '<th style="padding:7px 10px;color:#ffd600;text-align:left">Ticker</th>' +
            '<th style="color:#ffd600">OCC</th><th style="color:#ffd600">Close</th>' +
            '<th style="color:#ffd600;text-align:left;padding-left:10px">Failed Check(s)</th>' +
            '<th style="color:#ffd600;text-align:left">Sector</th>' +
            '<th style="color:#ffd600">Earnings</th>' +
            f"</tr></thead><tbody>{w_rows}</tbody></table>" +
            '<p style="color:#555;font-size:11px">Full watchlist in attached stage2_WATCHLIST CSV.</p>'
        )

    watch_count = len(watchlist_df) if watchlist_df is not None else 0
    dollar_risk = cfg.ACCOUNT_SIZE * cfg.RISK_PER_TRADE_PCT / 100

    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8">
<style>
  body{{font-family:Arial,sans-serif;background:#0d0d0d;color:#e0e0e0;margin:0;padding:20px}}
  .hdr{{background:linear-gradient(135deg,#1a1a2e,#16213e);border-radius:8px;padding:20px 28px;margin-bottom:18px}}
  .hdr h1{{margin:0;font-size:22px;color:#fff;letter-spacing:1px}}
  .hdr p{{margin:4px 0 0;color:#aaa;font-size:13px}}
  .stats{{display:flex;gap:12px;margin-bottom:18px;flex-wrap:wrap}}
  .stat{{background:#1a1a2e;border:1px solid #2a2a4e;border-radius:6px;padding:12px 18px;min-width:100px}}
  .stat .num{{font-size:22px;font-weight:bold;color:#4fc3f7}}
  .stat .lbl{{font-size:11px;color:#aaa;margin-top:2px}}
  table{{width:100%;border-collapse:collapse;font-size:12px}}
  th{{background:#0d2137;color:#4fc3f7;padding:8px 10px;text-align:left;font-size:11px;text-transform:uppercase;letter-spacing:.5px}}
  td{{padding:5px 10px;border-bottom:1px solid #111}}
  .pos-note{{background:#0d2137;border-radius:6px;padding:10px 16px;margin:8px 0 12px;font-size:11px;color:#aaa}}
  .footer{{margin-top:18px;font-size:11px;color:#555;border-top:1px solid #222;padding-top:10px}}
</style></head><body>

<div class="hdr">
  <h1>\U0001f402 BrokeBull Stage-2 Engine</h1>
  <p>Run Date: {today} &nbsp;|&nbsp; v2.3 &nbsp;|&nbsp; 40W MA + RS + Base Breakout + Position Sizing + Options</p>
  <p style="color:#4fc3f7;font-size:11px;margin-top:6px">
    Universe: NASDAQ · Other Listed · Russell 1000/2000/3000 · DJ Wilshire 5000 · MSCI World/EAFE · Nikkei 225
  </p>
</div>

<div class="stats">
  <div class="stat"><div class="num">{len(tech_df)}</div><div class="lbl">TECH</div></div>
  <div class="stat"><div class="num">{len(quality_df)}</div><div class="lbl">QUALITY</div></div>
  <div class="stat"><div class="num">{len(product_df)}</div><div class="lbl">PRODUCT</div></div>
  <div class="stat"><div class="num">{watch_count}</div><div class="lbl">WATCHLIST</div></div>
  <div class="stat"><div class="num">${cfg.ACCOUNT_SIZE:,.0f}</div><div class="lbl">Account</div></div>
  <div class="stat"><div class="num">{cfg.RISK_PER_TRADE_PCT}%</div><div class="lbl">Risk/Trade</div></div>
  <div class="stat"><div class="num">${dollar_risk:,.0f}</div><div class="lbl">$/Trade Risk</div></div>
</div>

<h3 style="color:#4fc3f7;margin:0 0 6px;font-size:14px">\U0001f3af TOP PRODUCT PICKS</h3>
<div class="pos-note">
  Position sizing: {cfg.RISK_PER_TRADE_PCT}% risk/trade = ${dollar_risk:,.0f} risked &nbsp;|&nbsp;
  Stop = {cfg.POSITION_STOP_ATR_MULT}\u00d7 ATR below entry &nbsp;|&nbsp;
  Pos $ and Stop shown per row below
</div>

<table>
  <thead><tr>
    <th>Ticker</th><th>Signal</th><th>OCC</th><th>Close</th>
    <th>ATR%</th><th>RelVol</th><th>BrkDays</th><th>Change</th><th>Held</th>
    <th>Earnings</th>
    <th style="color:#7ecfff">Pos $</th><th style="color:#7ecfff">Pos %</th>
    <th style="color:#ff8a80">Stop</th><th>Sector</th>
  </tr></thead>
  <tbody>{prod_rows}</tbody>
</table>

{sector_html}
{watch_html}

<div class="footer">
  Config: Weekly={cfg.STAGE2_WEEKLY_MA_WEEKS}W &nbsp;|&nbsp; Daily SMA={cfg.DAILY_SMA_DAYS}
  (req={cfg.REQUIRE_DAILY_200SMA_CONFIRM}) &nbsp;|&nbsp; Floor={cfg.PRODUCT_MIN_TIER} &nbsp;|&nbsp;
  Workers={cfg.SCORE_WORKERS}<br>
  FMP Fallback: {'enabled' if cfg.FMP_ENABLED and cfg.FMP_API_KEY else 'disabled'} &nbsp;|&nbsp;
  Cache: {cfg.CACHE_DIR}<br>
  Universe: NASDAQ · Other Listed · Russell 1000/2000/3000 · DJ Wilshire 5000 · MSCI World/EAFE · Nikkei 225<br>
  Options: SPY/QQQ/IWM + Mag 7 (calls & puts, OTM/ATM, {OPT_MIN_DTE}-{OPT_MAX_DTE} DTE)<br>
  Attachments: QUALITY.csv · PRODUCT.csv · WATCHLIST.csv · OPTIONS.csv
</div>
</body></html>"""
    return html


def send_email_with_attachments(
    cfg: CFG,
    subject: str,
    body_plain: str,
    attachments: List[str],
    body_html: str = "",
) -> Tuple[bool, str]:
    smtp_user = os.getenv("SMTP_USER", cfg.SMTP_USER).strip()
    smtp_pass = os.getenv("SMTP_PASS", cfg.SMTP_PASS).strip()
    email_to  = os.getenv("EMAIL_TO",  cfg.EMAIL_TO).strip()
    email_from= os.getenv("EMAIL_FROM",cfg.EMAIL_FROM).strip() or smtp_user
    smtp_host = os.getenv("SMTP_HOST", cfg.SMTP_HOST).strip()
    smtp_port = int(os.getenv("SMTP_PORT", str(cfg.SMTP_PORT)))

    if not smtp_user or not smtp_pass or not email_to:
        missing = []
        if not smtp_user: missing.append("SMTP_USER")
        if not smtp_pass: missing.append("SMTP_PASS")
        if not email_to:  missing.append("EMAIL_TO")
        return False, f"Missing credentials: {', '.join(missing)}"

    recipients = _parse_recipients(email_to)
    if not recipients:
        return False, "EMAIL_TO parsed to empty list — check format."

    from email.mime.multipart import MIMEMultipart
    from email.mime.text import MIMEText
    from email.mime.base import MIMEBase
    from email import encoders

    msg = MIMEMultipart("mixed")
    msg["From"]    = email_from
    msg["To"]      = ", ".join(recipients)
    msg["Subject"] = subject

    alt = MIMEMultipart("alternative")
    alt.attach(MIMEText(body_plain, "plain", "utf-8"))
    if body_html:
        alt.attach(MIMEText(body_html, "html", "utf-8"))
    msg.attach(alt)

    attached, skipped = [], []
    for p in attachments:
        if not p:
            continue
        if not os.path.exists(p):
            skipped.append(f"{os.path.basename(p)} (not found)")
            continue
        try:
            with open(p, "rb") as f:
                data = f.read()
            part = MIMEBase("application", "octet-stream")
            part.set_payload(data)
            encoders.encode_base64(part)
            part.add_header("Content-Disposition", "attachment",
                            filename=os.path.basename(p))
            msg.attach(part)
            attached.append(os.path.basename(p))
        except Exception as e:
            skipped.append(f"{os.path.basename(p)} ({e})")

    ctx = ssl.create_default_context()
    try:
        if smtp_port == 587:
            with smtplib.SMTP(smtp_host, smtp_port, timeout=45) as s:
                s.ehlo(); s.starttls(context=ctx); s.ehlo()
                s.login(smtp_user, smtp_pass)
                s.sendmail(email_from, recipients, msg.as_string())
        else:
            with smtplib.SMTP_SSL(smtp_host, smtp_port, context=ctx, timeout=45) as s:
                s.login(smtp_user, smtp_pass)
                s.sendmail(email_from, recipients, msg.as_string())

        note = f"Sent to {len(recipients)} recipient(s). Attached: {len(attached)} file(s)."
        if skipped: note += f" Skipped: {'; '.join(skipped)}"
        if attached: note += f" Files: {', '.join(attached)}"
        return True, note

    except smtplib.SMTPAuthenticationError:
        return False, (
            "SMTP Authentication failed.\n"
            "  → Gmail: make sure you're using an App Password (not your login password).\n"
            "  → Go to: myaccount.google.com → Security → App Passwords\n"
            "  → Generate a 16-char password and paste it into SMTP_PASS (no spaces)."
        )
    except smtplib.SMTPRecipientsRefused as e:
        return False, f"Recipient(s) refused: {e}"
    except smtplib.SMTPException as e:
        return False, f"SMTP error: {e}"
    except Exception as e:
        return False, f"Unexpected error: {type(e).__name__}: {e}"


# ══════════════════════════════════════════════════════════════════════════════
# UNIVERSE  (v2.2 — 12 sources: NASDAQ + Other + S&P 500/400/600 + Russell
#            1000/2000/3000 + DJ Wilshire 5000 + MSCI World/EAFE + Nikkei 225)
# ══════════════════════════════════════════════════════════════════════════════

# ── Existing NASDAQ sources (unchanged) ───────────────────────────────────────
NASDAQ_LISTED = "https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt"
OTHER_LISTED  = "https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt"
_EXCLUDE_NAME_PAT = re.compile(r"\b(?:ETF|ETN|WARRANT|RIGHTS?|UNIT(?:S)?|PFD|PREF|PREFERRED)\b", re.I)

# ── iShares ETF holdings URLs (public CSV downloads, no auth required) ──
_ISHARES_HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; research-bot/1.0)",
    "Accept": "text/csv,text/plain,*/*",
}

# S&P indices (v2.2)
_SP500_URL = (
    "https://www.ishares.com/us/products/239726/ishares-core-sp-500-etf"
    "/1467271812596.ajax?fileType=csv&fileName=IVV_holdings&dataType=fund"
)
_SP_MIDCAP_URL = (
    "https://www.ishares.com/us/products/239763/ishares-sp-midcap-400-etf"
    "/1467271812596.ajax?fileType=csv&fileName=IJH_holdings&dataType=fund"
)
_SP_SMALLCAP_URL = (
    "https://www.ishares.com/us/products/239774/ishares-sp-smallcap-600-etf"
    "/1467271812596.ajax?fileType=csv&fileName=IJR_holdings&dataType=fund"
)

# Russell indices
_RUSSELL_1000_URL = (
    "https://www.ishares.com/us/products/239707/ishares-russell-1000-etf"
    "/1467271812596.ajax?fileType=csv&fileName=IWB_holdings&dataType=fund"
)
_RUSSELL_2000_URL = (
    "https://www.ishares.com/us/products/239710/ishares-russell-2000-etf"
    "/1467271812596.ajax?fileType=csv&fileName=IWM_holdings&dataType=fund"
)
_RUSSELL_3000_URL = (
    "https://www.ishares.com/us/products/239714/ishares-russell-3000-etf"
    "/1467271812596.ajax?fileType=csv&fileName=IWV_holdings&dataType=fund"
)

# Dow Jones Wilshire 5000 — closest public proxy is iShares Core S&P Total US (ITOT)
_DJ_WILSHIRE_URL = (
    "https://www.ishares.com/us/products/239724/ishares-core-sp-total-us-stock-market-etf"
    "/1467271812596.ajax?fileType=csv&fileName=ITOT_holdings&dataType=fund"
)

# MSCI indices
_MSCI_WORLD_URL = (
    "https://www.ishares.com/us/products/239696/ishares-msci-acwi-etf"
    "/1467271812596.ajax?fileType=csv&fileName=ACWI_holdings&dataType=fund"
)
_MSCI_EAFE_URL = (
    "https://www.ishares.com/us/products/239623/ishares-msci-eafe-etf"
    "/1467271812596.ajax?fileType=csv&fileName=EFA_holdings&dataType=fund"
)

# Nikkei 225 — iShares MSCI Japan ETF (EWJ)
_NIKKEI_URL = (
    "https://www.ishares.com/us/products/239665/ishares-msci-japan-etf"
    "/1467271812596.ajax?fileType=csv&fileName=EWJ_holdings&dataType=fund"
)


def _fetch_ishares_holdings(url: str, label: str) -> List[str]:
    """Download iShares ETF holdings CSV, return cleaned equity tickers."""
    try:
        r = requests.get(url, timeout=60, headers=_ISHARES_HEADERS)
        r.raise_for_status()
        lines = r.text.splitlines()
        header_idx = None
        for i, line in enumerate(lines):
            if re.search(r'(?i)\bTicker\b', line):
                header_idx = i
                break
        if header_idx is None:
            print(f"[Universe/{label}] WARNING: 'Ticker' column not found in CSV — skipped.")
            return []
        csv_text = "\n".join(lines[header_idx:])
        df = pd.read_csv(io.StringIO(csv_text), dtype=str, on_bad_lines="skip")
        ticker_col = next((c for c in df.columns if c.strip().lower() == "ticker"), None)
        if ticker_col is None:
            print(f"[Universe/{label}] WARNING: No 'Ticker' column after parse — skipped.")
            return []
        syms = df[ticker_col].dropna().astype(str).str.strip().str.upper()
        syms = syms[
            syms.str.match(r'^[A-Z]{1,5}(\.[A-Z]{1,2})?$') |
            syms.str.match(r'^\d{4,5}\.[A-Z]{1,2}$')
        ]
        syms = syms[~syms.isin(["-", "N/A", "CASH", "USD", "EUR", "JPY", "GBP"])]
        result = syms.unique().tolist()
        print(f"[Universe/{label}] {len(result)} equity tickers loaded.")
        return result
    except requests.exceptions.RequestException as e:
        print(f"[Universe/{label}] Network error: {e} — skipped.")
        return []
    except Exception as e:
        print(f"[Universe/{label}] Parse error: {e} — skipped.")
        return []


def _fetch_russell_wikipedia_fallback(index: str) -> List[str]:
    wiki_urls = {
        "1000": "https://en.wikipedia.org/wiki/Russell_1000_Index",
        "2000": "https://en.wikipedia.org/wiki/Russell_2000_Index",
        "3000": "https://en.wikipedia.org/wiki/Russell_3000_Index",
    }
    url = wiki_urls.get(index)
    if not url:
        return []
    try:
        tables = pd.read_html(url, attrs=None)
        for t in tables:
            for col in t.columns:
                col_lower = str(col).lower()
                if "ticker" in col_lower or "symbol" in col_lower:
                    syms = t[col].dropna().astype(str).str.strip().str.upper()
                    syms = syms[syms.str.match(r'^[A-Z]{1,5}$')]
                    if len(syms) > 50:
                        result = syms.unique().tolist()
                        print(f"[Universe/Russell{index}/Wikipedia] {len(result)} tickers (fallback).")
                        return result
    except Exception as e:
        print(f"[Universe/Russell{index}/Wikipedia] Fallback error: {e}")
    return []


def _fetch_nikkei_wikipedia_fallback() -> List[str]:
    try:
        tables = pd.read_html("https://en.wikipedia.org/wiki/Nikkei_225")
        for t in tables:
            for col in t.columns:
                col_lower = str(col).lower()
                if "code" in col_lower or "ticker" in col_lower or "symbol" in col_lower:
                    syms = t[col].dropna().astype(str).str.strip()
                    numeric = syms[syms.str.match(r'^\d{4}$')]
                    if len(numeric) > 50:
                        result = [f"{s}.T" for s in numeric.unique().tolist()]
                        print(f"[Universe/Nikkei225/Wikipedia] {len(result)} tickers (fallback).")
                        return result
    except Exception as e:
        print(f"[Universe/Nikkei225/Wikipedia] Fallback error: {e}")
    return []


def _fetch_extended_index_tickers(cfg: CFG) -> Dict[str, List[str]]:
    """Fetch all requested extended indices in parallel."""
    tasks: List[Tuple[str, str, Optional[str]]] = []

    if cfg.UNIVERSE_INCLUDE_SP:
        tasks += [
            ("sp500",        _SP500_URL,      None),
            ("sp_midcap_400", _SP_MIDCAP_URL,  None),
            ("sp_smallcap_600", _SP_SMALLCAP_URL, None),
        ]
    if cfg.UNIVERSE_INCLUDE_RUSSELL:
        tasks += [
            ("russell_1000", _RUSSELL_1000_URL, "1000"),
            ("russell_2000", _RUSSELL_2000_URL, "2000"),
            ("russell_3000", _RUSSELL_3000_URL, "3000"),
        ]
    if cfg.UNIVERSE_INCLUDE_WILSHIRE:
        tasks += [("dj_wilshire_5000", _DJ_WILSHIRE_URL, None)]
    if cfg.UNIVERSE_INCLUDE_MSCI:
        tasks += [
            ("msci_world_acwi", _MSCI_WORLD_URL, None),
            ("msci_eafe",       _MSCI_EAFE_URL,  None),
        ]
    if cfg.UNIVERSE_INCLUDE_NIKKEI:
        tasks += [("nikkei_225", _NIKKEI_URL, None)]

    if not tasks:
        print("[Universe] All extended indices disabled — using NASDAQ/Other only.")
        return {}

    results: Dict[str, List[str]] = {}

    def _fetch_one(task: Tuple[str, str, Optional[str]]) -> Tuple[str, List[str]]:
        label, url, fallback_key = task
        tickers = _fetch_ishares_holdings(url, label)
        if not tickers:
            if fallback_key and label.startswith("russell"):
                tickers = _fetch_russell_wikipedia_fallback(fallback_key)
            elif label == "nikkei_225":
                tickers = _fetch_nikkei_wikipedia_fallback()
        return label, tickers

    with ThreadPoolExecutor(max_workers=cfg.UNIVERSE_WORKERS) as ex:
        for label, tickers in ex.map(_fetch_one, tasks):
            results[label] = tickers
            if not tickers:
                print(f"[Universe/{label}] WARNING: 0 tickers returned — check URL or fallback.")

    return results


def _fetch_symbol_file(url: str) -> pd.DataFrame:
    r = requests.get(url, timeout=60); r.raise_for_status()
    lines = [l for l in r.text.strip().splitlines() if not l.startswith("File Creation Time")]
    return pd.read_csv(io.StringIO("\n".join(lines)), sep="|")


def _clean_nasdaq(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"Security Name": "SecurityName"})
    for col, val in [("Test Issue","Y"), ("ETF","Y"), ("NextShares","Y")]:
        if col in df.columns:
            df = df[df[col].astype(str).str.upper() != val]
    if "Financial Status" in df.columns:
        df = df[df["Financial Status"].fillna("").isin(["", "N"])]
    return df[["Symbol", "SecurityName"]]


def _clean_other(df: pd.DataFrame) -> pd.DataFrame:
    df = df.rename(columns={"ACT Symbol": "Symbol", "Security Name": "SecurityName"})
    for col, val in [("Test Issue","Y"), ("ETF","Y")]:
        if col in df.columns:
            df = df[df[col].astype(str).str.upper() != val]
    return df[["Symbol", "SecurityName"]]


def _apply_symbol_exclusions(sym: pd.Series) -> pd.Series:
    s = sym.astype(str).str.strip()
    s = s[~s.str.contains(r"[=:^ ]", regex=True, na=False)]
    s = s[~s.str.contains(r"\$", regex=True, na=False)]
    s = s[~s.str.contains(r"(?:\.(?:W|WS|WT|U|R)$)|(?:-(?:W|WS|WT|U|R)$)", regex=True, na=False)]
    return s


def build_universe_for_yahoo(cfg: CFG = None) -> Tuple[List[str], Dict[str, str]]:
    """Build the full ticker universe from 12 sources, deduplicated."""
    if cfg is None:
        cfg = CFG()

    print("[Universe] Fetching NASDAQ Listed + Other Listed …")
    nas = _clean_nasdaq(_fetch_symbol_file(NASDAQ_LISTED))
    oth = _clean_other(_fetch_symbol_file(OTHER_LISTED))
    nas_sym = _apply_symbol_exclusions(nas["Symbol"])
    oth_sym = _apply_symbol_exclusions(oth["Symbol"])
    base_full = pd.concat([nas, oth], ignore_index=True).drop_duplicates("Symbol")
    base_full = base_full[base_full["Symbol"].isin(pd.concat([nas_sym, oth_sym]).astype(str))]
    base_full = base_full[~base_full["SecurityName"].fillna("").str.contains(_EXCLUDE_NAME_PAT, regex=True)]
    nasdaq_syms: set = set(base_full["Symbol"].astype(str).str.strip().tolist())
    print(f"[Universe] NASDAQ/Other: {len(nasdaq_syms)} symbols")

    print("[Universe] Fetching extended index constituents (S&P/Russell/Wilshire/MSCI/Nikkei) …")
    index_map = _fetch_extended_index_tickers(cfg)

    index_syms: set = set()
    for label, raw_tickers in index_map.items():
        clean: set = set()
        for t in raw_tickers:
            t = str(t).strip().upper()
            if re.match(r'^[A-Z]{1,6}(\.[A-Z]{1,2})?$', t) or re.match(r'^\d{4,5}\.[A-Z]{1,2}$', t):
                if t not in ("-", "N/A"):
                    clean.add(t)
        index_syms |= clean
        print(f"[Universe]   {label}: {len(clean)} symbols after clean")

    all_syms = nasdaq_syms | index_syms
    new_via_indices = index_syms - nasdaq_syms
    print(
        f"\n[Universe] ─── TOTAL ──────────────────────────────────\n"
        f"[Universe]   NASDAQ/Other Listed : {len(nasdaq_syms):>6,}\n"
        f"[Universe]   Extended indices    : {len(index_syms):>6,} unique\n"
        f"[Universe]   New via indices     : {len(new_via_indices):>6,} (not in NASDAQ)\n"
        f"[Universe]   Grand total         : {len(all_syms):>6,} (after dedup)\n"
        f"[Universe] ───────────────────────────────────────────────\n"
    )

    syms = sorted(all_syms)
    ymap: Dict[str, str] = {}
    for s in syms:
        if re.match(r'^[A-Z]{1,5}\.[A-Z]$', s):
            ymap[s] = s.replace(".", "-")
        else:
            ymap[s] = s

    return syms, ymap


# ══════════════════════════════════════════════════════════════════════════════
# PRICE FETCHING — batched download (used by cache layer below)
# ══════════════════════════════════════════════════════════════════════════════

def _chunks(lst: List[str], n: int):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]


def _normalize_ohlcv(df: pd.DataFrame) -> pd.DataFrame:
    if "Date" in df.columns:
        df = df.rename(columns={"Date": "date"})
    df = df.rename(columns={"Open":"open","High":"high","Low":"low","Close":"close","Volume":"volume"})
    keep = [c for c in ["date","open","high","low","close","volume"] if c in df.columns]
    df = df[keep].dropna()
    if not {"date","high","low","close","volume"}.issubset(set(df.columns)):
        return pd.DataFrame()
    df["date"] = pd.to_datetime(df["date"])
    return df


def _download_batch_raw(yahoo_symbols: List[str], rev_map: Dict[str, List[str]],
                        period: str, yf_threads: bool, chunk_size: int,
                        out_dir: str, today: str) -> Tuple[Dict[str, pd.DataFrame], List[str]]:
    out: Dict[str, pd.DataFrame] = {}
    bad: List[str] = []

    for batch in _chunks(yahoo_symbols, chunk_size):
        try:
            df = yf.download(tickers=batch, period=period, interval="1d",
                             group_by="ticker", auto_adjust=True,
                             threads=yf_threads, progress=False)
        except Exception as e:
            print(f"[Download] Batch error: {e}")
            bad.extend(batch); continue

        if df is None or df.empty:
            bad.extend(batch); continue

        if isinstance(df.columns, pd.MultiIndex):
            if df.columns.levels[0].isin(["Open","High","Low","Close","Volume"]).any():
                df = df.swaplevel(axis=1)
            for y in batch:
                if y not in df.columns.get_level_values(0):
                    bad.append(y); continue
                sub = df[y].copy()
                if not {"Close","Volume","High","Low"}.issubset(set(sub.columns)):
                    bad.append(y); continue
                sub = _normalize_ohlcv(sub.reset_index())
                if sub.empty:
                    bad.append(y); continue
                for orig in rev_map.get(y, []):
                    out[orig] = sub
        else:
            if {"Close","Volume","High","Low"}.issubset(set(df.columns)):
                sub = _normalize_ohlcv(df.reset_index())
                y = batch[0]
                if sub.empty:
                    bad.append(y)
                else:
                    for orig in rev_map.get(y, []):
                        out[orig] = sub
            else:
                bad.append(batch[0])

    if bad:
        os.makedirs(out_dir, exist_ok=True)
        pd.DataFrame({"yahoo_symbol": sorted(bad)}).to_csv(
            os.path.join(out_dir, f"bad_yahoo_symbols_{today}.csv"), index=False)
        print(f"[Download] Bad symbols: {len(bad)}")

    return out, bad


def fetch_prices_cached(tickers: List[str], ymap: Dict[str, str], cfg: CFG) -> Dict[str, pd.DataFrame]:
    """REC #1 — Parquet-cached price fetcher."""
    os.makedirs(cfg.CACHE_DIR, exist_ok=True)
    os.makedirs(cfg.OUT_DIR, exist_ok=True)
    today = dt.date.today().isoformat()
    parquet_path, meta_path = _cache_paths(cfg.CACHE_DIR, cfg.PERIOD_STR, today)

    all_tickers = list(dict.fromkeys(tickers + ["SPY"]))
    all_yahoo   = [ymap.get(t, t) for t in all_tickers]
    rev_map: Dict[str, List[str]] = {}
    for t, y in zip(all_tickers, all_yahoo):
        rev_map.setdefault(y, []).append(t)

    if not cfg.FORCE_PRICE_REFRESH:
        cached_df, meta = _load_cache(parquet_path, meta_path)
        if cached_df is not None and meta is not None:
            cached_set = set(meta.get("tickers", []))
            missing    = set(all_tickers) - cached_set
            print(f"[Cache] HIT — {len(cached_set)} cached | {len(missing)} new")
            prices = _wide_to_dict(cached_df)

            if missing:
                miss_yahoo = [ymap.get(t, t) for t in missing]
                miss_rev: Dict[str, List[str]] = {}
                for t, y in zip(list(missing), miss_yahoo):
                    miss_rev.setdefault(y, []).append(t)
                new_prices, new_bad = _download_batch_raw(
                    miss_yahoo, miss_rev, cfg.PERIOD_STR,
                    cfg.YF_THREADS, cfg.YF_CHUNK, cfg.OUT_DIR, today)
                prices.update(new_prices)
                if new_prices:
                    merged = pd.concat([cached_df, _dict_to_long(new_prices)], ignore_index=True)
                    merged = merged.drop_duplicates(subset=["ticker","date"])
                    updated_meta = {
                        "date": today, "period": cfg.PERIOD_STR,
                        "fetch_timestamp": dt.datetime.now().isoformat(),
                        "tickers": sorted(set(meta.get("tickers",[])) | set(new_prices.keys())),
                        "ticker_count": len(merged["ticker"].unique()),
                        "bad_symbols": meta.get("bad_symbols",[]) + new_bad,
                    }
                    _save_cache(merged, updated_meta, parquet_path, meta_path)
            return prices

    action = "FORCE REFRESH" if cfg.FORCE_PRICE_REFRESH else "MISS"
    print(f"[Cache] {action} — downloading {len(all_tickers)} tickers …")
    prices, bad = _download_batch_raw(
        all_yahoo, rev_map, cfg.PERIOD_STR,
        cfg.YF_THREADS, cfg.YF_CHUNK, cfg.OUT_DIR, today)

    if prices:
        long_df = _dict_to_long(prices)
        meta = {
            "date": today, "period": cfg.PERIOD_STR,
            "fetch_timestamp": dt.datetime.now().isoformat(),
            "tickers": sorted(prices.keys()),
            "ticker_count": len(prices),
            "bad_symbols": bad,
        }
        _save_cache(long_df, meta, parquet_path, meta_path)

    print(f"[Cache] Complete: {len(prices)} fetched, {len(bad)} failed")
    return prices


# ══════════════════════════════════════════════════════════════════════════════
# REC #6 — DEDUPLICATED INDICATOR HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def sma(series: pd.Series, window: int) -> pd.Series:
    return series.rolling(window=window, min_periods=window).mean()


def compute_weekly_close(df: pd.DataFrame) -> pd.Series:
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date")
    s = pd.to_numeric(d["close"], errors="coerce")
    wk = pd.DataFrame({"date": d["date"], "close": s}).dropna()
    wk = wk.resample("W-FRI", on="date").last()
    wk["close"] = wk["close"].ffill()
    return wk["close"].dropna()


def daily_close_series(df: pd.DataFrame) -> pd.Series:
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date")
    s = pd.to_numeric(d["close"], errors="coerce")
    s.index = d["date"].values
    return s.dropna()


def weekly_ohlcv_from_daily(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date").dropna(subset=["close","high","low","volume"])
    d["close"]  = pd.to_numeric(d["close"],  errors="coerce")
    d["volume"] = pd.to_numeric(d["volume"], errors="coerce")
    wk = d.resample("W-FRI", on="date").agg(
        {"close":"last","high":"max","low":"min","volume":"sum"}).dropna()
    wk["close"] = wk["close"].ffill()
    return wk.rename(columns={"close":"Close","high":"High","low":"Low","volume":"Volume"})


# ══════════════════════════════════════════════════════════════════════════════
# STAGE-2 GATES
# ══════════════════════════════════════════════════════════════════════════════

def stage2_weekly_gate(weekly_close: pd.Series, cfg: CFG) -> dict:
    wk = weekly_close
    need = cfg.STAGE2_WEEKLY_MA_WEEKS + cfg.STAGE2_WEEKLY_SLOPE_LOOKBACK_WK + 1
    if len(wk) < need:
        return {"stage2_weekly_ok": False}
    ma_w = sma(wk, cfg.STAGE2_WEEKLY_MA_WEEKS)
    if ma_w.isna().all():
        return {"stage2_weekly_ok": False}
    last_px  = float(wk.iloc[-1])
    last_ma  = float(ma_w.iloc[-1])
    prev_ma  = float(ma_w.iloc[-1 - cfg.STAGE2_WEEKLY_SLOPE_LOOKBACK_WK])
    price_above = last_px > last_ma
    ma_rising   = last_ma > prev_ma
    return {
        "close_weekly": last_px,
        "stage2_weekly_ma_weeks": int(cfg.STAGE2_WEEKLY_MA_WEEKS),
        "stage2_weekly_ma": float(last_ma),
        "price_above_stage2_weekly_ma": bool(price_above),
        "stage2_weekly_ma_rising": bool(ma_rising),
        "stage2_weekly_ok": bool(price_above and ma_rising),
    }


def stage2_daily_200sma_confirm(df: pd.DataFrame, cfg: CFG) -> dict:
    dclose = daily_close_series(df)
    need = cfg.DAILY_SMA_DAYS + cfg.DAILY_SLOPE_LOOKBACK_DAYS + 5
    if len(dclose) < need:
        return {"daily_200sma_ok": False}
    ma_d = sma(dclose, cfg.DAILY_SMA_DAYS)
    if ma_d.isna().all():
        return {"daily_200sma_ok": False}
    last_px  = float(dclose.iloc[-1])
    last_ma  = float(ma_d.iloc[-1])
    prev_ma  = float(ma_d.iloc[-1 - cfg.DAILY_SLOPE_LOOKBACK_DAYS])
    price_above = last_px > last_ma
    ma_rising   = last_ma > prev_ma
    return {
        "close_daily": last_px,
        "daily_sma_days": int(cfg.DAILY_SMA_DAYS),
        "daily_sma": float(last_ma),
        "price_above_daily_sma": bool(price_above),
        "daily_sma_rising": bool(ma_rising),
        "daily_200sma_ok": bool(price_above and ma_rising),
    }


def rs_vs_spy_features(stock_weekly: pd.Series, spy_df: pd.DataFrame, cfg: CFG) -> dict:
    s   = stock_weekly
    spy = compute_weekly_close(spy_df)
    idx = s.index.intersection(spy.index)
    if len(idx) < max(cfg.RS_MA_WEEKS, cfg.RS_SLOPE_LOOKBACK_WK) + 5:
        return {"rs_ok": False}
    rs = (s.loc[idx] / spy.loc[idx]).replace([np.inf, -np.inf], np.nan).dropna()
    if len(rs) < max(cfg.RS_MA_WEEKS, cfg.RS_SLOPE_LOOKBACK_WK) + 5:
        return {"rs_ok": False}
    rs_ma    = sma(rs, cfg.RS_MA_WEEKS)
    rs_last  = float(rs.iloc[-1])
    rs_ma_last = float(rs_ma.iloc[-1]) if math.isfinite(rs_ma.iloc[-1]) else float("nan")
    rs_past  = float(rs.iloc[-1 - cfg.RS_SLOPE_LOOKBACK_WK])
    rs_slope = rs_last - rs_past
    rs_above_ma = math.isfinite(rs_ma_last) and rs_last > rs_ma_last
    rs_rising   = rs_slope > 0
    rs_ok = True
    if cfg.RS_REQUIRE_ABOVE_MA: rs_ok = rs_ok and rs_above_ma
    if cfg.RS_REQUIRE_RISING:   rs_ok = rs_ok and rs_rising
    vol     = float(rs.diff().rolling(13).std().iloc[-1])
    slope_z = rs_slope / vol if (math.isfinite(vol) and vol > 0) else rs_slope
    rs_score = float(np.clip(50 + 15 * slope_z, 0, 100))
    return {"rs_ok": bool(rs_ok), "rs_last": rs_last, "rs_ma": rs_ma_last,
            "rs_slope": rs_slope, "rs_score": rs_score}


def base_breakout_check(daily: pd.DataFrame, weekly_close: pd.Series, cfg: CFG) -> dict:
    d = daily.copy()
    d["date"] = pd.to_datetime(d["date"])
    d = d.sort_values("date").dropna(subset=["close","volume","high","low"])
    if len(d) < 260:
        return {"base_ok": False}
    avg50 = float(d["volume"].tail(50).mean())
    if not (math.isfinite(avg50) and avg50 > 0):
        return {"base_ok": False}
    lookback = int(cfg.BREAKOUT_LOOKBACK_DAYS)
    if lookback < 1 or len(d) <= lookback + 30:
        return {"base_ok": False}

    max_base_days = cfg.BASE_MAX_WEEKS * 5
    d["roll_high"] = d["high"].rolling(window=max_base_days, min_periods=1).max()
    d["roll_low"]  = d["low"].rolling(window=max_base_days, min_periods=1).min()

    recent = d.iloc[-lookback:].copy()
    base_end_idx = len(d) - lookback
    last_close   = float(d["close"].iloc[-1])
    min_d = cfg.BASE_MIN_WEEKS * 5
    max_d = cfg.BASE_MAX_WEEKS * 5
    best  = None

    for L in range(min_d, max_d + 1, 5):
        base_slice = d.iloc[max(0, base_end_idx - L): base_end_idx]
        if len(base_slice) < min_d:
            continue
        base_high = float(base_slice["high"].max())
        base_low  = float(base_slice["low"].min())
        if not (math.isfinite(base_high) and math.isfinite(base_low)) or base_high <= 0:
            continue
        depth = (base_high - base_low) / base_high
        if depth > cfg.BASE_MAX_DEPTH:
            continue
        pivot       = base_high * (1.0 + cfg.BREAKOUT_BUFFER_PCT)
        recent_break = recent[recent["close"] > pivot]
        if recent_break.empty:
            continue
        bd      = recent_break.iloc[0]
        bvol    = float(bd["volume"])
        vol_mult = bvol / avg50
        if not (math.isfinite(vol_mult) and vol_mult >= cfg.BREAKOUT_MIN_VOL_MULT):
            continue
        if last_close < base_high * cfg.RETEST_FLOOR_PCT:
            continue

        wk_slice = weekly_close[weekly_close.index >= base_slice["date"].iloc[0]]
        wk_slice = wk_slice[wk_slice.index <= base_slice["date"].iloc[-1]]
        tight = float(wk_slice.pct_change().abs().median()) if len(wk_slice) > 10 else float("nan")

        score = 0.0
        if math.isfinite(tight):   score += max(0, 30 - 600 * tight)
        score += max(0, 40 - 120 * depth)
        score += 15
        score += 15

        breakout_date = pd.to_datetime(bd["date"]) if "date" in bd.index else None
        days_since_bo = (dt.date.today() - breakout_date.date()).days if breakout_date else float("nan")

        cand = {
            "base_ok": True,
            "base_high": float(base_high), "base_low": float(base_low),
            "base_depth": float(depth),    "base_len_days": int(L),
            "base_tight": float(tight),    "breakout_vol_mult": float(vol_mult),
            "base_score": float(np.clip(score, 0, 100)),
            "days_since_breakout": days_since_bo,
        }
        if best is None or cand["base_score"] > best["base_score"]:
            best = cand

    return best if best is not None else {"base_ok": False}


def trend_template_score(daily: pd.DataFrame, cfg: CFG) -> dict:
    d = daily.copy().dropna(subset=["close"])
    close = pd.to_numeric(d["close"], errors="coerce").dropna()
    if len(close) < max(cfg.TREND_MA_SLOW, cfg.TREND_52W_DAYS) + 10:
        return {"trend_score": 0.0, "trend_ok": False}
    ma50  = sma(close, cfg.TREND_MA_FAST)
    ma150 = sma(close, cfg.TREND_MA_MID)
    ma200 = sma(close, cfg.TREND_MA_SLOW)
    last = float(close.iloc[-1]); last50 = float(ma50.iloc[-1])
    last150 = float(ma150.iloc[-1]); last200 = float(ma200.iloc[-1])
    past200 = float(ma200.iloc[-21]) if len(ma200)>21 and math.isfinite(ma200.iloc[-21]) else float("nan")
    ma200_rising = math.isfinite(past200) and last200 > past200
    w = close.tail(cfg.TREND_52W_DAYS)
    low52 = float(w.min()); high52 = float(w.max())
    dist_from_low  = (last - low52)  / low52  if low52  > 0 else float("nan")
    dist_from_high = (high52 - last) / high52 if high52 > 0 else float("nan")
    score = 0.0
    score += 15 if (last > last150 and last > last200) else 0
    score += 15 if (last150 > last200)                 else 0
    score += 10 if ma200_rising                         else 0
    score += 15 if (last50 > last150 > last200)         else 0
    score += 10 if (last > last50)                      else 0
    score += 10 if (last > (low52 + 0.5*(high52-low52))) else 0
    score += 15 if (math.isfinite(dist_from_high) and dist_from_high <= 0.25) else 0
    score += 10 if (math.isfinite(dist_from_low)  and dist_from_low  >= 0.30) else 0
    return {
        "trend_score": float(score), "trend_ok": bool(score >= cfg.TREND_OK_SCORE),
        "ma50": float(ma50.iloc[-1]), "ma150": float(ma150.iloc[-1]),
        "ma200": float(ma200.iloc[-1]), "high52": high52, "low52": low52,
    }


# ══════════════════════════════════════════════════════════════════════════════
# REC #5 — ADDITIONAL OUTPUT COLUMNS
# ══════════════════════════════════════════════════════════════════════════════

def compute_extra_features(daily: pd.DataFrame) -> dict:
    d = daily.copy().dropna(subset=["high","low","close","volume"])
    d["close"]  = pd.to_numeric(d["close"],  errors="coerce")
    d["high"]   = pd.to_numeric(d["high"],   errors="coerce")
    d["low"]    = pd.to_numeric(d["low"],    errors="coerce")
    d["volume"] = pd.to_numeric(d["volume"], errors="coerce")
    d = d.dropna()
    out = {}
    if len(d) >= 15:
        prev_close = d["close"].shift(1)
        tr = pd.concat([
            d["high"] - d["low"],
            (d["high"] - prev_close).abs(),
            (d["low"]  - prev_close).abs(),
        ], axis=1).max(axis=1)
        atr14     = float(tr.rolling(14).mean().iloc[-1])
        last_close = float(d["close"].iloc[-1])
        out["atr_pct"] = round(atr14 / last_close * 100, 3) if last_close > 0 else float("nan")
    else:
        out["atr_pct"] = float("nan")
    if len(d) >= 21:
        vol_20d_avg = float(d["volume"].iloc[-21:-1].mean())
        today_vol   = float(d["volume"].iloc[-1])
        out["rel_volume"] = round(today_vol / vol_20d_avg, 2) if vol_20d_avg > 0 else float("nan")
    else:
        out["rel_volume"] = float("nan")
    return out


def fetch_earnings_dates(tickers: List[str], max_workers: int = 6) -> Dict[str, Optional[str]]:
    def _get(sym: str) -> Tuple[str, Optional[str]]:
        for attempt in range(3):
            try:
                cal = yf.Ticker(sym).get_calendar()
                if cal is not None and not cal.empty:
                    for col in cal.columns:
                        val = cal[col].iloc[0] if len(cal) > 0 else None
                        if val is not None and hasattr(val, "date"):
                            return sym, str(val.date())
                return sym, None
            except Exception:
                time.sleep(2 ** attempt)
        return sym, None

    result: Dict[str, Optional[str]] = {}
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for sym, date_str in ex.map(lambda t: _get(t), tickers):
            result[sym] = date_str
    return result


# ══════════════════════════════════════════════════════════════════════════════
# OCC SCORING
# ══════════════════════════════════════════════════════════════════════════════

def _to_f(x):
    try:
        v = float(x); return v if math.isfinite(v) else float("nan")
    except Exception:
        return float("nan")

def _pct(x):
    v = _to_f(x)
    if not math.isfinite(v): return float("nan")
    return v * 100.0 if 0 < v < 3 else v

def _clamp(x, lo=0.0, hi=1.0):
    if x is None or (isinstance(x, float) and not math.isfinite(x)): return float("nan")
    return max(lo, min(hi, x))

def _score_linear(x, x0, x1):
    if x is None or (isinstance(x, float) and not math.isfinite(x)): return float("nan")
    if x1 == x0: return 1.0 if x >= x1 else 0.0
    return _clamp((x - x0) / (x1 - x0), 0.0, 1.0)


def compute_technical_score_from_features(stage2_weekly: dict, rs: dict, base: dict, trend: dict) -> dict:
    price_above = 1.0 if stage2_weekly.get("price_above_stage2_weekly_ma") else 0.0
    ma_rising   = 1.0 if stage2_weekly.get("stage2_weekly_ma_rising")      else 0.0
    t_trend_pts = 10*price_above + 10*ma_rising
    rs_score  = _to_f(rs.get("rs_score")); rs_slope = _to_f(rs.get("rs_slope"))
    rs_lvl_s  = _score_linear(rs_score, 50.0, 80.0)
    rs_slope_s = 1.0 if (math.isfinite(rs_slope) and rs_slope > 0) else 0.0
    t_rs_pts   = 10*(rs_lvl_s if math.isfinite(rs_lvl_s) else 0) + 10*rs_slope_s
    base_score = _to_f(base.get("base_score")); depth = _to_f(base.get("base_depth"))
    tight = _to_f(base.get("base_tight"))
    base_lvl_s = _score_linear(base_score, 55.0, 85.0)
    depth_s    = 1.0 - _score_linear(depth, 0.10, 0.35)
    tight_s    = 1.0 - _score_linear(tight, 0.015, 0.060)
    base_q     = np.nanmean([base_lvl_s, depth_s, tight_s])
    t_base_pts = 20*(base_q if math.isfinite(base_q) else 0)
    vol_mult = _to_f(base.get("breakout_vol_mult"))
    vol_s = _score_linear(vol_mult, 1.0, 2.0)
    t_breakout_pts = 20*(vol_s if math.isfinite(vol_s) else 0)
    trend_score = _to_f(trend.get("trend_score"))
    trend_s = _score_linear(trend_score, 55.0, 90.0)
    t_template_pts = 20*(trend_s if math.isfinite(trend_s) else 0)
    technical_score = float(t_trend_pts + t_rs_pts + t_base_pts + t_breakout_pts + t_template_pts)
    return {
        "technical_score": round(technical_score, 2),
        "t_trend_pts": round(t_trend_pts, 2), "t_rs_pts": round(t_rs_pts, 2),
        "t_base_pts":  round(t_base_pts, 2),  "t_breakout_pts": round(t_breakout_pts, 2),
        "t_template_pts": round(t_template_pts, 2),
    }


def compute_risk_score_from_daily(daily_df: pd.DataFrame) -> dict:
    wk = weekly_ohlcv_from_daily(daily_df)
    if len(wk) < 40:
        return {"risk_score": float("nan")}
    close = wk["Close"]
    wret  = close.pct_change().dropna()
    wvol  = float(wret.std()) if len(wret) else float("nan")
    roll_max = close.cummax()
    mdd = float(((close / roll_max) - 1.0).min())
    d   = daily_df.copy().dropna(subset=["close","volume"])
    d["close"]  = pd.to_numeric(d["close"],  errors="coerce")
    d["volume"] = pd.to_numeric(d["volume"], errors="coerce")
    d   = d.dropna()
    adv = float((d["close"].tail(30) * d["volume"].tail(30)).mean()) if len(d) >= 30 else float("nan")
    vol_s  = 1.0 - _score_linear(wvol, 0.02, 0.08)
    mdd_s  = 1.0 - _score_linear(abs(mdd), 0.10, 0.45)
    liq_s  = _score_linear(adv, 10e6, 250e6)
    pen_s  = 1.0 - _score_linear(wvol, 0.06, 0.12)
    risk_score = float(
        30*(vol_s if math.isfinite(vol_s) else 0) +
        30*(mdd_s if math.isfinite(mdd_s) else 0) +
        20*(liq_s if math.isfinite(liq_s) else 0) +
        20*(pen_s if math.isfinite(pen_s) else 0)
    )
    return {"risk_score": round(risk_score, 2), "max_drawdown": mdd, "avg_dollar_vol_30d": adv}


THEME_MAP = {
    "Technology": 0.75, "Industrials": 0.65, "Energy": 0.70,
    "Communication Services": 0.62, "Financial Services": 0.58,
    "Healthcare": 0.58, "Consumer Cyclical": 0.60,
    "Consumer Defensive": 0.55, "Basic Materials": 0.60,
    "Utilities": 0.50, "Real Estate": 0.50,
}

def compute_macro_score_from_row(r: pd.Series) -> dict:
    sector   = (r.get("sector") or "").strip()
    industry = (r.get("industry") or "").strip()
    base = THEME_MAP.get(sector, 0.55)
    if any(k in industry for k in ["Defense","Aerospace"]):   base = max(base, 0.80)
    if any(k in industry for k in ["Semiconductor","Data Center"]): base = max(base, 0.78)
    if any(k in industry for k in ["Oil","Pipeline"]):         base = max(base, 0.72)
    return {"macro_score": round(float(100.0 * _clamp(base, 0.0, 1.0)), 2)}


def compute_occ_score(tech, fund, risk, macro,
                      w_tech=0.40, w_fund=0.30, w_risk=0.15, w_macro=0.15) -> float:
    def neutral(x):
        try:
            x = float(x); return 50.0 if not math.isfinite(x) else x
        except Exception:
            return 50.0
    t, f, r, m = neutral(tech), neutral(fund), neutral(risk), neutral(macro)
    return round(w_tech*t + w_fund*f + w_risk*r + w_macro*m, 2)


def occ_tier(score: float) -> str:
    try:
        s = float(score)
        if not math.isfinite(s): return "NO_DATA"
    except Exception:
        return "NO_DATA"
    if s >= 90: return "TIER_1_CORE"
    if s >= 80: return "TIER_2_HIGH"
    if s >= 70: return "TIER_3_WATCH"
    return "TIER_4_REJECT"


def occ_signal(tier: str) -> str:
    return {
        "TIER_1_CORE":    "STRONG BUY",
        "TIER_2_HIGH":    "BUY",
        "TIER_3_WATCH":   "WATCH",
        "TIER_4_REJECT":  "AVOID",
        "NO_DATA":        "NO DATA",
    }.get(tier, "UNKNOWN")


# ══════════════════════════════════════════════════════════════════════════════
# FUNDAMENTALS
# ══════════════════════════════════════════════════════════════════════════════

FUND_COLUMNS = [
    "sector","industry","pe","eps","roe_pct","peg","ps","pb",
    "ev_ebitda","ev_sales","net_margin_pct","op_margin_pct",
    "fcf_yield_pct","debt_to_equity","interest_coverage"
]


def fetch_yahoo_info(tickers: List[str], max_workers: int = 6) -> pd.DataFrame:
    if not tickers:
        return pd.DataFrame(columns=["ticker"] + FUND_COLUMNS)

    def _get_info_with_retry(sym: str) -> dict:
        for attempt in range(3):
            try:
                return yf.Ticker(sym).get_info() or {}
            except Exception:
                if attempt < 2:
                    time.sleep(2 ** attempt)
        return {}

    def _y(sym: str) -> dict:
        out = {"ticker": sym}
        info = _get_info_with_retry(sym)
        if not info:
            return out
        out["sector"]        = info.get("sector") or ""
        out["industry"]      = info.get("industry") or ""
        out["pe"]            = _to_f(info.get("trailingPE"))
        out["ps"]            = _to_f(info.get("priceToSalesTrailing12Months"))
        out["pb"]            = _to_f(info.get("priceToBook"))
        out["ev_ebitda"]     = _to_f(info.get("enterpriseToEbitda"))
        out["ev_sales"]      = _to_f(info.get("enterpriseToRevenue"))
        out["net_margin_pct"]= _pct(info.get("profitMargins"))
        out["op_margin_pct"] = _pct(info.get("operatingMargins"))
        out["roe_pct"]       = _pct(info.get("returnOnEquity"))
        out["debt_to_equity"]= _to_f(info.get("debtToEquity"))
        out["interest_coverage"] = float("nan")
        out["eps"]           = _to_f(info.get("trailingEps"))
        out["peg"]           = _to_f(info.get("pegRatio"))
        fcf  = _to_f(info.get("freeCashflow"))
        mcap = _to_f(info.get("marketCap"))
        out["fcf_yield_pct"] = (fcf/mcap*100.0) if (math.isfinite(fcf) and math.isfinite(mcap) and mcap > 0) else float("nan")
        return out

    rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for r in ex.map(_y, list(dict.fromkeys(tickers))):
            rows.append(r)

    df = pd.DataFrame(rows)
    for c in FUND_COLUMNS:
        if c not in df.columns:
            df[c] = "" if c in ("industry","sector") else np.nan
    return df[["ticker"] + FUND_COLUMNS]


def _compute_fundamental_score_from_row_safe(r: pd.Series) -> dict:
    eps = _to_f(r.get("eps")); roe = _to_f(r.get("roe_pct"))
    netm = _to_f(r.get("net_margin_pct")); pe = _to_f(r.get("pe"))
    peg  = _to_f(r.get("peg")); ev = _to_f(r.get("ev_ebitda"))
    fcfy = _to_f(r.get("fcf_yield_pct")); de = _to_f(r.get("debt_to_equity"))
    eps_s  = _score_linear(eps, 0.0, 8.0)
    f_eps_pts  = 20*(eps_s if math.isfinite(eps_s) else 0)
    roe_s  = _score_linear(roe, 8.0, 25.0); netm_s = _score_linear(netm, 0.0, 20.0)
    prof_s = np.nanmean([roe_s, netm_s])
    f_prof_pts = 25*(prof_s if math.isfinite(prof_s) else 0)
    pe_s   = 1.0 - _score_linear(pe, 12.0, 40.0)
    peg_s  = 1.0 - _score_linear(peg, 1.0, 3.0)
    ev_s   = 1.0 - _score_linear(ev, 8.0, 22.0)
    val_s  = np.nanmean([pe_s, peg_s, ev_s])
    f_val_pts  = 25*(val_s if math.isfinite(val_s) else 0)
    de_s   = 1.0 - _score_linear(de, 0.3, 2.0)
    f_bs_pts   = 15*(de_s if math.isfinite(de_s) else 0)
    fcf_s  = _score_linear(fcfy, 1.0, 6.0)
    f_cf_pts   = 15*(fcf_s if math.isfinite(fcf_s) else 0)
    return {"fundamental_score": round(float(f_eps_pts+f_prof_pts+f_val_pts+f_bs_pts+f_cf_pts), 2)}


def valuation_pass_flags(row: pd.Series, cfg: CFG) -> dict:
    t = cfg.VAL_THRESHOLDS
    def ok(v, pred, thr, allow_missing=False):
        if not math.isfinite(v): return (not cfg.STRICT_VALUATION) or allow_missing
        return pred(v, thr)
    r = row.to_dict()
    flags = {
        "val_eps_pos":        ok(_to_f(r.get("eps")),              lambda v,m: v  >  m, t["MIN_EPS_TTM"]),
        "val_roe":            ok(_to_f(r.get("roe_pct")),          lambda v,m: v  >= m, t["MIN_ROE_PCT"]),
        "val_net_margin":     ok(_to_f(r.get("net_margin_pct")),   lambda v,m: v  >  m, t["MIN_NET_MARGIN_PCT"]),
        "val_pe":             ok(_to_f(r.get("pe")),               lambda v,m: 0<v<=m,  t["MAX_PE"]),
        "val_peg":            ok(_to_f(r.get("peg")),              lambda v,m: 0<v<=m,  t["MAX_PEG"]),
        "val_pb":             ok(_to_f(r.get("pb")),               lambda v,m: 0<v<=m,  t["MAX_PB"]),
        "val_ps":             ok(_to_f(r.get("ps")),               lambda v,m: 0<v<=m,  t["MAX_PS"]),
        "val_ev_ebitda":      ok(_to_f(r.get("ev_ebitda")),        lambda v,m: 0<v<=m,  t["MAX_EV_EBITDA"]),
        "val_fcf_yield":      ok(_to_f(r.get("fcf_yield_pct")),    lambda v,m: v  >= m, t["MIN_FCF_YIELD_PCT"]),
        "val_debt_to_equity": ok(_to_f(r.get("debt_to_equity")),   lambda v,m: v  <= m, t["MAX_DEBT_TO_EQUITY"]),
        "val_int_coverage":   ok(_to_f(r.get("interest_coverage")), lambda v,m: v >= m, t["MIN_INTEREST_COVERAGE"], allow_missing=True),
    }
    flags["valuation_ok"] = all(flags.values())
    return flags


def apply_sector_relative_filter(df: pd.DataFrame, cfg: CFG) -> pd.DataFrame:
    if df.empty or not cfg.RELATIVE_FILTER_ENABLED:
        df["rel_ok"] = True; return df
    out = df.copy()
    rel = cfg.REL_THRESHOLDS

    def sector_median(col):
        return out.groupby("industry", dropna=False)[col].transform("median")
    for col in ["pe","ev_ebitda","peg","ps","pb","net_margin_pct","roe_pct","fcf_yield_pct"]:
        if col in out.columns:
            out[f"{col}_sector_med"] = sector_median(col)

    def lt_mult(col, mult):
        med = out.get(f"{col}_sector_med"); v = out.get(col)
        if med is None or v is None: return pd.Series(True, index=out.index)
        return (v <= (mult*med)) | (~np.isfinite(v)) | (~np.isfinite(med))

    def gt_minus(col, minus_pp):
        med = out.get(f"{col}_sector_med"); v = out.get(col)
        if med is None or v is None: return pd.Series(True, index=out.index)
        return (v >= (med+minus_pp)) | (~np.isfinite(v)) | (~np.isfinite(med))

    out["rel_pe_ok"]        = lt_mult("pe",          rel["PE_MAX_MULT"])
    out["rel_ev_ebitda_ok"] = lt_mult("ev_ebitda",   rel["EV_EBITDA_MAX_MULT"])
    out["rel_peg_ok"]       = lt_mult("peg",          rel["PEG_MAX_MULT"])
    out["rel_ps_ok"]        = lt_mult("ps",           rel["PS_MAX_MULT"])
    out["rel_pb_ok"]        = lt_mult("pb",           rel["PB_MAX_MULT"])
    out["rel_netm_ok"]      = gt_minus("net_margin_pct", rel["NET_MARGIN_MIN_PC"])
    out["rel_roe_ok"]       = gt_minus("roe_pct",        rel["ROE_MIN_PC"])
    out["rel_fcf_ok"]       = gt_minus("fcf_yield_pct",  rel["FCF_YIELD_MIN_PC"])
    out["rel_ok"] = out[["rel_pe_ok","rel_ev_ebitda_ok","rel_peg_ok","rel_ps_ok","rel_pb_ok",
                          "rel_netm_ok","rel_roe_ok","rel_fcf_ok"]].all(axis=1)
    return out


# ══════════════════════════════════════════════════════════════════════════════
# REC #3 — DELTA / CHANGE TRACKING
# ══════════════════════════════════════════════════════════════════════════════

def load_yesterday_product(out_dir: str, today: str) -> Optional[pd.DataFrame]:
    yesterday = (dt.date.today() - dt.timedelta(days=1)).isoformat()
    for candidate_date in [yesterday]:
        path = os.path.join(out_dir, f"stage2_PRODUCT_{candidate_date}.csv")
        if os.path.exists(path):
            try:
                return pd.read_csv(path)
            except Exception:
                pass
    pattern = "stage2_PRODUCT_"
    candidates = sorted([
        f for f in os.listdir(out_dir)
        if f.startswith(pattern) and f.endswith(".csv") and today not in f
    ], reverse=True)
    for fname in candidates:
        try:
            return pd.read_csv(os.path.join(out_dir, fname))
        except Exception:
            continue
    return None


def compute_delta(today_df: pd.DataFrame, yesterday_df: Optional[pd.DataFrame],
                  today: str, out_dir: str) -> pd.DataFrame:
    if today_df.empty:
        return today_df

    df = today_df.copy()
    df["first_seen"] = today
    df["last_seen"]  = today
    df["days_in_product"] = 1
    df["signal_change"]   = "new"

    if yesterday_df is None or yesterday_df.empty:
        print("[Delta] No previous PRODUCT found — all entries marked 'new'.")
        return df

    prev = yesterday_df.copy()
    prev_tickers  = set(prev["ticker"].astype(str))
    today_tickers = set(df["ticker"].astype(str))
    new_entries  = today_tickers - prev_tickers
    dropped      = prev_tickers  - today_tickers
    continued    = today_tickers & prev_tickers

    if "first_seen" in prev.columns:
        prev_map = prev.set_index("ticker")[["first_seen", "days_in_product"]].to_dict("index")
    else:
        prev_map = {}

    prev_tier_map = prev.set_index("ticker")["occ_tier"].to_dict() if "occ_tier" in prev.columns else {}

    def get_change(row):
        t = str(row["ticker"])
        if t in new_entries:
            return "new"
        prev_tier  = prev_tier_map.get(t, "")
        curr_tier  = row.get("occ_tier", "")
        tier_order = {"TIER_1_CORE":4,"TIER_2_HIGH":3,"TIER_3_WATCH":2,"TIER_4_REJECT":1,"NO_DATA":0}
        prev_rank  = tier_order.get(prev_tier, 0)
        curr_rank  = tier_order.get(curr_tier, 0)
        if curr_rank > prev_rank:   return "tier_upgrade"
        if curr_rank < prev_rank:   return "tier_downgrade"
        return "unchanged"

    df["signal_change"] = df.apply(get_change, axis=1)

    for i, row in df.iterrows():
        t = str(row["ticker"])
        if t in prev_map:
            p = prev_map[t]
            df.at[i, "first_seen"]    = p.get("first_seen", today)
            existing_days = p.get("days_in_product", 1)
            try:
                existing_days = int(existing_days)
            except Exception:
                existing_days = 1
            df.at[i, "days_in_product"] = existing_days + 1

    delta_rows = []
    for t in sorted(new_entries):
        row = df[df["ticker"]==t].iloc[0]
        delta_rows.append({"ticker":t,"change":"new","occ_tier":row.get("occ_tier",""),"occ_score":row.get("occ_score","")})
    for t in sorted(dropped):
        p_row = prev[prev["ticker"]==t].iloc[0] if len(prev[prev["ticker"]==t]) else {}
        delta_rows.append({"ticker":t,"change":"dropped","occ_tier":p_row.get("occ_tier","") if isinstance(p_row,pd.Series) else "","occ_score":""})
    for t in sorted(continued):
        row = df[df["ticker"]==t].iloc[0]
        change = row["signal_change"]
        if change not in ("unchanged",):
            delta_rows.append({"ticker":t,"change":change,"occ_tier":row.get("occ_tier",""),"occ_score":row.get("occ_score","")})

    if delta_rows:
        delta_df   = pd.DataFrame(delta_rows)
        delta_path = os.path.join(out_dir, f"stage2_DELTA_{today}.csv")
        delta_df.to_csv(delta_path, index=False)
        print(f"[Delta] New:{len(new_entries)} | Dropped:{len(dropped)} | "
              f"Upgrades:{sum(1 for r in delta_rows if r['change']=='tier_upgrade')} | "
              f"Downgrades:{sum(1 for r in delta_rows if r['change']=='tier_downgrade')} → {delta_path}")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE 3 — POSITION SIZING
# ══════════════════════════════════════════════════════════════════════════════

def compute_position_size(atr_pct: float, close: float, cfg: CFG) -> dict:
    try:
        atr_pct = float(atr_pct); close = float(close)
        if not (math.isfinite(atr_pct) and math.isfinite(close) and atr_pct > 0 and close > 0):
            return {"pos_shares": float("nan"), "pos_dollars": float("nan"),
                    "pos_pct_account": float("nan"), "stop_price": float("nan")}
        dollar_risk    = cfg.ACCOUNT_SIZE * (cfg.RISK_PER_TRADE_PCT / 100.0)
        stop_distance  = close * (atr_pct / 100.0) * cfg.POSITION_STOP_ATR_MULT
        shares         = dollar_risk / stop_distance
        pos_dollars    = shares * close
        pos_pct        = (pos_dollars / cfg.ACCOUNT_SIZE) * 100.0
        stop_price     = close - stop_distance
        return {
            "pos_shares": round(shares, 0), "pos_dollars": round(pos_dollars, 2),
            "pos_pct_account": round(pos_pct, 2), "stop_price": round(stop_price, 2),
            "dollar_risk": round(dollar_risk, 2),
        }
    except Exception:
        return {"pos_shares": float("nan"), "pos_dollars": float("nan"),
                "pos_pct_account": float("nan"), "stop_price": float("nan"),
                "dollar_risk": float("nan")}


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE 4 — FMP FALLBACK DATA SOURCE
# ══════════════════════════════════════════════════════════════════════════════

FMP_BASE = "https://financialmodelingprep.com/api/v3"

def _fmp_get(endpoint: str, api_key: str, timeout: int = 10) -> Optional[dict]:
    try:
        url = f"{FMP_BASE}/{endpoint}?apikey={api_key}"
        r   = requests.get(url, timeout=timeout)
        if r.status_code == 200:
            data = r.json()
            if isinstance(data, list) and data: return data[0]
            if isinstance(data, dict): return data
        return None
    except Exception:
        return None


def fetch_fmp_fundamentals(tickers: List[str], api_key: str,
                           timeout: int = 10, max_workers: int = 6) -> pd.DataFrame:
    if not api_key or not tickers:
        return pd.DataFrame(columns=["ticker"] + FUND_COLUMNS)

    def _fetch_one(sym: str) -> dict:
        out = {"ticker": sym}
        ratios = _fmp_get(f"ratios-ttm/{sym}", api_key, timeout)
        if ratios:
            out["pe"]           = _to_f(ratios.get("peRatioTTM"))
            out["pb"]           = _to_f(ratios.get("priceToBookRatioTTM"))
            out["ps"]           = _to_f(ratios.get("priceToSalesRatioTTM"))
            out["peg"]          = _to_f(ratios.get("priceEarningsToGrowthRatioTTM"))
            out["ev_ebitda"]    = _to_f(ratios.get("enterpriseValueMultipleTTM"))
            out["roe_pct"]      = _pct(ratios.get("returnOnEquityTTM"))
            out["net_margin_pct"]= _pct(ratios.get("netProfitMarginTTM"))
            out["op_margin_pct"]= _pct(ratios.get("operatingProfitMarginTTM"))
            out["debt_to_equity"]= _to_f(ratios.get("debtEquityRatioTTM"))
            out["interest_coverage"] = _to_f(ratios.get("interestCoverageTTM"))
            fcf_yield = _to_f(ratios.get("freeCashFlowYieldTTM"))
            out["fcf_yield_pct"] = fcf_yield * 100.0 if (math.isfinite(fcf_yield) and abs(fcf_yield) < 3) else fcf_yield
        profile = _fmp_get(f"profile/{sym}", api_key, timeout)
        if profile:
            out["eps"]      = _to_f(profile.get("eps"))
            out["sector"]   = profile.get("sector") or ""
            out["industry"] = profile.get("industry") or ""
            out["ev_sales"] = _to_f(profile.get("enterpriseValue"))
        return out

    rows = []
    with ThreadPoolExecutor(max_workers=max_workers) as ex:
        for r in ex.map(_fetch_one, list(dict.fromkeys(tickers))):
            rows.append(r)

    df = pd.DataFrame(rows) if rows else pd.DataFrame(columns=["ticker"] + FUND_COLUMNS)
    for c in FUND_COLUMNS:
        if c not in df.columns:
            df[c] = "" if c in ("industry", "sector") else np.nan
    return df[["ticker"] + FUND_COLUMNS]


def fetch_fundamentals_with_fallback(tickers: List[str], cfg: CFG) -> pd.DataFrame:
    print(f"Fetching fundamentals: Yahoo ({len(tickers)} tickers) …")
    yahoo_df = fetch_yahoo_info(tickers, max_workers=cfg.FUND_MAX_WORKERS)

    if not (cfg.FMP_ENABLED and cfg.FMP_API_KEY):
        print("FMP fallback disabled (FMP_API_KEY not set) — using Yahoo only.")
        return yahoo_df

    critical = ["pe", "eps", "roe_pct"]
    missing_mask = yahoo_df[critical].isna().all(axis=1)
    missing_tickers = yahoo_df.loc[missing_mask, "ticker"].tolist()

    if not missing_tickers:
        print("FMP fallback: Yahoo covered all tickers — no fallback needed.")
        return yahoo_df

    print(f"FMP fallback: {len(missing_tickers)} tickers missing Yahoo data → fetching from FMP …")
    fmp_df = fetch_fmp_fundamentals(
        missing_tickers, cfg.FMP_API_KEY, cfg.FMP_TIMEOUT, cfg.FUND_MAX_WORKERS)

    combined = yahoo_df.set_index("ticker").copy()
    fmp_idx  = fmp_df.set_index("ticker")

    for t in missing_tickers:
        if t in fmp_idx.index:
            for col in FUND_COLUMNS:
                if col in ("sector", "industry"):
                    if not combined.at[t, col]:
                        combined.at[t, col] = fmp_idx.at[t, col]
                else:
                    if col in combined.columns and col in fmp_idx.columns:
                        if not math.isfinite(_to_f(combined.at[t, col])):
                            combined.at[t, col] = fmp_idx.at[t, col]

    result = combined.reset_index()
    fmp_filled = result.loc[result["ticker"].isin(missing_tickers), critical].notna().any(axis=1).sum()
    print(f"FMP fallback: filled data for {fmp_filled}/{len(missing_tickers)} previously empty tickers.")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE 2 — WATCHLIST
# ══════════════════════════════════════════════════════════════════════════════

def build_watchlist(tech_df: pd.DataFrame, quality_df: pd.DataFrame,
                    cfg: CFG, today: str) -> pd.DataFrame:
    if tech_df.empty:
        return pd.DataFrame()
    tech_tickers    = set(tech_df["ticker"].astype(str))
    quality_tickers = set(quality_df["ticker"].astype(str)) if not quality_df.empty else set()
    watch_tickers   = tech_tickers - quality_tickers
    if not watch_tickers:
        print("Watchlist: all TECH tickers passed QUALITY — no watchlist candidates.")
        return pd.DataFrame()
    watch_df = tech_df[tech_df["ticker"].isin(watch_tickers)].copy()
    val_flag_cols = [c for c in quality_df.columns if c.startswith("val_")] if not quality_df.empty else []
    for col in val_flag_cols:
        if col not in watch_df.columns:
            watch_df[col] = float("nan")

    def failed_flags(row):
        fails = []
        for col in val_flag_cols:
            val = row.get(col)
            if val is False or val == 0:
                fails.append(col.replace("val_", ""))
        return ", ".join(fails) if fails else "sector_relative_filter"

    if val_flag_cols:
        watch_df["failed_checks"] = watch_df.apply(failed_flags, axis=1)
    else:
        watch_df["failed_checks"] = "valuation_filter"
    watch_df["watchlist_added"] = today
    watch_df = watch_df.sort_values(
        ["occ_score", "composite_score", "ticker"], ascending=[False, False, True]
    ).reset_index(drop=True).head(cfg.WATCHLIST_MAX_ROWS)
    watch_cols = [
        "ticker", "signal", "occ_tier", "occ_score", "composite_score",
        "technical_score", "fundamental_score", "risk_score",
        "close", "atr_pct", "rel_volume", "days_since_breakout", "earnings_date",
        "failed_checks", "watchlist_added",
        "sector", "industry", "pe", "peg", "pb", "ps",
        "ev_ebitda", "fcf_yield_pct", "roe_pct", "debt_to_equity",
        "rs_score", "base_score",
    ]
    watch_df = watch_df.reindex(columns=[c for c in watch_cols if c in watch_df.columns])
    print(f"Watchlist: {len(watch_df)} candidates (TECH pass / QUALITY fail)")
    return watch_df


# ══════════════════════════════════════════════════════════════════════════════
# UPGRADE 1 — SECTOR BREAKDOWN FOR EMAIL
# ══════════════════════════════════════════════════════════════════════════════

def build_sector_breakdown(product_df: pd.DataFrame, watchlist_df: pd.DataFrame) -> dict:
    result = {"product": [], "watchlist": []}
    for label, df in [("product", product_df), ("watchlist", watchlist_df)]:
        if df is None or df.empty or "sector" not in df.columns:
            continue
        grp = df.groupby("sector", dropna=False).agg(
            count=("ticker", "count"),
            avg_occ=("occ_score", "mean"),
            tickers=("ticker", lambda x: ", ".join(sorted(x.astype(str).tolist())[:6])),
        ).reset_index().sort_values("count", ascending=False)
        grp["avg_occ"] = grp["avg_occ"].round(1)
        result[label] = grp.to_dict("records")
    return result


# ══════════════════════════════════════════════════════════════════════════════
# REC #2 — PARALLEL SCORING LOOP
# ══════════════════════════════════════════════════════════════════════════════

def _score_one_ticker(args) -> Optional[dict]:
    t, df, spy_df, cfg = args
    weekly_close = compute_weekly_close(df)
    s2w = stage2_weekly_gate(weekly_close, cfg)
    if not s2w.get("stage2_weekly_ok"):
        return None
    s2d = stage2_daily_200sma_confirm(df, cfg)
    if cfg.REQUIRE_DAILY_200SMA_CONFIRM and not s2d.get("daily_200sma_ok"):
        return None
    rs = rs_vs_spy_features(weekly_close, spy_df, cfg)
    if not rs.get("rs_ok"):
        return None
    base = base_breakout_check(df, weekly_close, cfg)
    if not base.get("base_ok"):
        return None
    trend = trend_template_score(df, cfg)
    if not trend.get("trend_ok"):
        return None
    composite = (
        0.40 * float(rs.get("rs_score", 0)) +
        0.35 * float(trend.get("trend_score", 0)) +
        0.25 * float(base.get("base_score", 0))
    )
    occ_tech = compute_technical_score_from_features(s2w, rs, base, trend)
    occ_risk = compute_risk_score_from_daily(df)
    extra = compute_extra_features(df)
    return {
        "ticker": t, "composite_score": round(float(composite), 2),
        "stage2_weekly_ma_weeks": int(s2w.get("stage2_weekly_ma_weeks", cfg.STAGE2_WEEKLY_MA_WEEKS)),
        "stage2_weekly_ma": float(s2w.get("stage2_weekly_ma", np.nan)),
        "price_above_stage2_weekly_ma": bool(s2w.get("price_above_stage2_weekly_ma")),
        "stage2_weekly_ma_rising": bool(s2w.get("stage2_weekly_ma_rising")),
        "daily_sma_days": int(s2d.get("daily_sma_days", cfg.DAILY_SMA_DAYS)),
        "daily_sma": float(s2d.get("daily_sma", np.nan)),
        "price_above_daily_sma": bool(s2d.get("price_above_daily_sma")),
        "daily_sma_rising": bool(s2d.get("daily_sma_rising")),
        "daily_200sma_ok": bool(s2d.get("daily_200sma_ok")),
        "close": float(df["close"].iloc[-1]),
        "rs_score": round(float(rs.get("rs_score", np.nan)), 2),
        "rs_slope": float(rs.get("rs_slope", np.nan)),
        "rs_last":  float(rs.get("rs_last",  np.nan)),
        "rs_ma":    float(rs.get("rs_ma",    np.nan)),
        "base_score": round(float(base.get("base_score", np.nan)), 2),
        "base_len_days": int(base.get("base_len_days", 0) or 0),
        "base_depth": float(base.get("base_depth", np.nan)),
        "base_tight": float(base.get("base_tight", np.nan)),
        "breakout_vol_mult": float(base.get("breakout_vol_mult", np.nan)),
        "base_high": float(base.get("base_high", np.nan)),
        "base_low":  float(base.get("base_low",  np.nan)),
        "days_since_breakout": base.get("days_since_breakout", float("nan")),
        "technical_score": occ_tech.get("technical_score", np.nan),
        "risk_score": occ_risk.get("risk_score", np.nan),
        "max_drawdown": occ_risk.get("max_drawdown", np.nan),
        "avg_dollar_vol_30d": occ_risk.get("avg_dollar_vol_30d", np.nan),
        "trend_score": round(float(trend.get("trend_score", np.nan)), 2),
        "trend_ok": bool(trend.get("trend_ok")),
        "atr_pct": extra.get("atr_pct", float("nan")),
        "rel_volume": extra.get("rel_volume", float("nan")),
        "_s2d_ok": bool(s2d.get("daily_200sma_ok")),
    }


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

def run_stage2_pipeline(cfg: CFG) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    os.makedirs(cfg.OUT_DIR, exist_ok=True)
    os.makedirs(cfg.CACHE_DIR, exist_ok=True)
    today = dt.date.today().isoformat()

    info = cache_info(cfg.CACHE_DIR)
    if not info.empty:
        print("[Cache Status]"); print(info.to_string(index=False)); print()

    tickers, ymap = build_universe_for_yahoo(cfg)
    print(f"Universe: {len(tickers)} symbols")

    prices = fetch_prices_cached(tickers, ymap, cfg)
    purge_old_cache(cfg.CACHE_DIR, keep_days=cfg.CACHE_KEEP_DAYS)

    print(f"Price histories: {len(prices)} (SPY={'SPY' in prices})")
    if "SPY" not in prices:
        raise RuntimeError("SPY missing — cannot compute RS vs SPY.")
    spy_df = prices["SPY"]

    liquid: Dict[str, pd.DataFrame] = {}
    for t, df in prices.items():
        if t == "SPY" or df is None or len(df) < 260:
            continue
        px  = float(df["close"].iloc[-1])
        v30 = float(df["volume"].tail(30).mean())
        if px >= cfg.MIN_PRICE and v30 >= cfg.MIN_AVG_VOL_30D:
            liquid[t] = df
    print(f"After liquidity gate: {len(liquid)}")

    gate = {"liquid": len(liquid), "stage2_weekly": 0, "stage2_daily_confirm": 0,
            "rs": 0, "base": 0, "trend_ok": 0}
    gate_lock = threading.Lock()
    rows: List[dict] = []
    rows_lock = threading.Lock()
    score_args = [(t, df, spy_df, cfg) for t, df in liquid.items()]

    print(f"Scoring {len(score_args)} liquid tickers with {cfg.SCORE_WORKERS} workers …")
    with ThreadPoolExecutor(max_workers=cfg.SCORE_WORKERS) as ex:
        for result in ex.map(_score_one_ticker, score_args):
            if result is not None:
                with gate_lock:
                    gate["stage2_weekly"]   += 1
                    gate["rs"]              += 1
                    gate["base"]            += 1
                    gate["trend_ok"]        += 1
                    if result.pop("_s2d_ok", False):
                        gate["stage2_daily_confirm"] += 1
                with rows_lock:
                    rows.append(result)

    print("Gate counts:", gate)

    stage2_df = pd.DataFrame(rows).sort_values(
        ["composite_score","ticker"], ascending=[False, True]
    ).reset_index(drop=True)

    raw_path  = os.path.join(cfg.OUT_DIR, f"stage2_RAW_{today}.csv")
    tech_path = os.path.join(cfg.OUT_DIR, f"stage2_TECH_{today}.csv")
    qual_path = os.path.join(cfg.OUT_DIR, f"stage2_QUALITY_{today}.csv")
    prod_path = os.path.join(cfg.OUT_DIR, cfg.PRODUCT_FILENAME or f"stage2_PRODUCT_{today}.csv")

    stage2_df.to_csv(raw_path, index=False)
    print(f"Wrote RAW: {raw_path}")

    if stage2_df.empty:
        for p in [tech_path, qual_path, prod_path]:
            stage2_df.to_csv(p, index=False)
        print("No passers — empty outputs written.")
        # Still run options scan even if stage 2 empty
        options_path = run_options_scan(cfg.OUT_DIR, today)
        return stage2_df, stage2_df, stage2_df, pd.DataFrame()

    fdf    = fetch_fundamentals_with_fallback(stage2_df["ticker"].tolist(), cfg)
    merged = stage2_df.merge(fdf.drop_duplicates("ticker"), on="ticker", how="left")

    print("Fetching earnings dates …")
    earnings_map = fetch_earnings_dates(merged["ticker"].tolist(), max_workers=cfg.FUND_MAX_WORKERS)
    merged["earnings_date"] = merged["ticker"].map(earnings_map)

    merged = pd.concat([
        merged.reset_index(drop=True),
        merged.apply(lambda r: pd.Series(_compute_fundamental_score_from_row_safe(r)), axis=1).reset_index(drop=True),
        merged.apply(lambda r: pd.Series(compute_macro_score_from_row(r)), axis=1).reset_index(drop=True),
    ], axis=1)

    merged["occ_score"] = merged.apply(
        lambda r: compute_occ_score(_to_f(r.get("technical_score")), _to_f(r.get("fundamental_score")),
                                    _to_f(r.get("risk_score")), _to_f(r.get("macro_score"))), axis=1)
    merged["occ_tier"]   = merged["occ_score"].apply(occ_tier)
    merged["signal"]     = merged["occ_tier"].apply(occ_signal)

    pos_cols = merged.apply(
        lambda r: pd.Series(compute_position_size(_to_f(r.get("atr_pct")), _to_f(r.get("close")), cfg)),
        axis=1
    ).reset_index(drop=True)
    merged = pd.concat([merged.reset_index(drop=True), pos_cols], axis=1)

    drop_debug = ["base_len_days","base_depth","base_tight","breakout_vol_mult",
                  "base_high","base_low","trend_score","trend_ok"]
    tech_cols = [
        "ticker","signal","occ_tier","occ_score","composite_score",
        "technical_score","fundamental_score","risk_score","macro_score",
        "close","atr_pct","rel_volume",
        "stage2_weekly_ma_weeks","stage2_weekly_ma",
        "price_above_stage2_weekly_ma","stage2_weekly_ma_rising",
        "daily_sma_days","daily_sma","price_above_daily_sma","daily_sma_rising","daily_200sma_ok",
        "rs_score","rs_slope","rs_last","rs_ma","base_score",
        "days_since_breakout","earnings_date",
    ] + FUND_COLUMNS

    tech_df = merged.drop(columns=[c for c in drop_debug if c in merged.columns], errors="ignore")
    tech_df = tech_df.reindex(columns=[c for c in tech_cols if c in tech_df.columns]).copy()
    tech_df = tech_df.sort_values(["occ_score","composite_score","ticker"],
                                   ascending=[False,False,True]).reset_index(drop=True)
    tech_df.to_csv(tech_path, index=False)
    print(f"Wrote TECH: {tech_path} (rows={len(tech_df)})")

    abs_flags = merged.apply(lambda r: pd.Series(valuation_pass_flags(r, cfg)), axis=1)
    with_abs  = pd.concat([merged.reset_index(drop=True), abs_flags], axis=1)
    with_rel  = apply_sector_relative_filter(with_abs, cfg)
    with_rel["final_ok"] = with_rel["valuation_ok"] & with_rel["rel_ok"]

    # ─── Funnel diagnostics (v2.3.1) ───────────────────────────────────────
    val_cols = [c for c in with_rel.columns if c.startswith("val_") and c != "valuation_ok"]
    rel_cols = [c for c in with_rel.columns if c.startswith("rel_") and c != "rel_ok"]
    print(f"\n[Funnel] Failure breakdown of {len(with_rel)} TECH passes:")
    print(f"  ── Absolute valuation checks ──")
    for col in val_cols:
        # Coerce to bool (some flags arrive as object dtype with NaN — can't ~ those directly)
        s = with_rel[col].fillna(True).astype(bool)
        failed = int((~s).sum())
        if failed > 0:
            pct = 100.0 * failed / max(1, len(with_rel))
            print(f"    {col:<28} failed: {failed:>4} ({pct:>5.1f}%)")
    print(f"  ── Sector-relative checks ──")
    for col in rel_cols:
        s = with_rel[col].fillna(True).astype(bool)
        failed = int((~s).sum())
        if failed > 0:
            pct = 100.0 * failed / max(1, len(with_rel))
            print(f"    {col:<28} failed: {failed:>4} ({pct:>5.1f}%)")
    n_val_ok = int(with_rel["valuation_ok"].fillna(False).astype(bool).sum())
    n_rel_ok = int(with_rel["rel_ok"].fillna(False).astype(bool).sum())
    n_both   = int((with_rel["valuation_ok"].fillna(False).astype(bool)
                    & with_rel["rel_ok"].fillna(False).astype(bool)).sum())
    print(f"  ─── valuation_ok={n_val_ok} | rel_ok={n_rel_ok} | BOTH (final QUALITY)={n_both}\n")
    # ────────────────────────────────────────────────────────────────────────

    quality_df = with_rel[with_rel["final_ok"]].copy()
    quality_df = quality_df.sort_values(["occ_score","composite_score","ticker"],
                                         ascending=[False,False,True]).reset_index(drop=True)
    quality_df.to_csv(qual_path, index=False)
    print(f"Wrote QUALITY: {qual_path} (rows={len(quality_df)})")

    tier_order   = {"TIER_1_CORE":4,"TIER_2_HIGH":3,"TIER_3_WATCH":2,"TIER_4_REJECT":1,"NO_DATA":0}
    min_tier_val = tier_order.get(cfg.PRODUCT_MIN_TIER, 2)
    product_df   = quality_df[
        quality_df["occ_tier"].map(tier_order).fillna(0) >= min_tier_val
    ].head(cfg.PRODUCT_MAX_ROWS).copy()

    if cfg.DELTA_ENABLED:
        yesterday_product = load_yesterday_product(cfg.OUT_DIR, today)
        product_df = compute_delta(product_df, yesterday_product, today, cfg.OUT_DIR)

    product_cols = [
        "ticker","signal","occ_tier","occ_score","composite_score",
        "technical_score","fundamental_score","risk_score","macro_score",
        "signal_change","days_in_product","first_seen","last_seen",
        "close","atr_pct","rel_volume","days_since_breakout","earnings_date",
        "pos_shares","pos_dollars","pos_pct_account","stop_price","dollar_risk",
        "stage2_weekly_ma","stage2_weekly_ma_rising",
        "daily_sma","daily_sma_rising","daily_200sma_ok",
        "sector","industry","pe","peg","pb","ps","ev_ebitda","fcf_yield_pct","roe_pct","debt_to_equity",
    ]
    product_df = product_df.reindex(columns=[c for c in product_cols if c in product_df.columns])
    product_df.to_csv(prod_path, index=False)
    print(f"Wrote PRODUCT: {prod_path} (rows={len(product_df)})")

    watch_path = os.path.join(cfg.OUT_DIR, cfg.WATCHLIST_FILENAME or f"stage2_WATCHLIST_{today}.csv")
    watchlist_df = pd.DataFrame()
    if cfg.WATCHLIST_ENABLED:
        tech_with_flags = merged.copy()
        for col in [c for c in with_rel.columns if c.startswith("val_")]:
            if col not in tech_with_flags.columns:
                tech_with_flags[col] = with_rel[col].values if len(with_rel) == len(tech_with_flags) else float("nan")
        watchlist_df = build_watchlist(tech_with_flags, quality_df, cfg, today)
        if not watchlist_df.empty:
            watchlist_df.to_csv(watch_path, index=False)
            print(f"Wrote WATCHLIST: {watch_path} (rows={len(watchlist_df)})")

    # ══════════════════════════════════════════════════════════════════════
    # OPTIONS LAYER (v2.3) — runs on fixed watchlist (SPY/QQQ/IWM + Mag 7)
    # ══════════════════════════════════════════════════════════════════════
    options_path = run_options_scan(cfg.OUT_DIR, today)

    sector_data = build_sector_breakdown(product_df, watchlist_df)

    email_ready, _ = diagnose_email_config(cfg)

    if email_ready:
        attachments: List[str] = []

        # ─── Attachment build diagnostics (v2.3.3) ────────────────────────
        print("\n[Email] Building attachment list:")

        attachments.append(qual_path)
        print(f"  ✓ QUALITY  : {qual_path} (always attached)")

        if cfg.EMAIL_ATTACH_PRODUCT:
            if os.path.exists(prod_path):
                attachments.append(prod_path)
                print(f"  ✓ PRODUCT  : {prod_path}")
            else:
                print(f"  ✗ PRODUCT  : SKIPPED — file not found at {prod_path}")
        else:
            print(f"  ○ PRODUCT  : disabled by EMAIL_ATTACH_PRODUCT=false")

        if cfg.EMAIL_ATTACH_TECH:
            attachments.append(tech_path)
            print(f"  ✓ TECH     : {tech_path}")
        else:
            print(f"  ○ TECH     : disabled by EMAIL_ATTACH_TECH=false")

        if cfg.EMAIL_ATTACH_WATCHLIST:
            if os.path.exists(watch_path):
                attachments.append(watch_path)
                print(f"  ✓ WATCHLIST: {watch_path}")
            else:
                print(f"  ✗ WATCHLIST: SKIPPED — file not found at {watch_path}")
        else:
            print(f"  ○ WATCHLIST: disabled by EMAIL_ATTACH_WATCHLIST=false")

        if cfg.EMAIL_ATTACH_OPTIONS:
            if not options_path:
                print(f"  ✗ OPTIONS  : SKIPPED — run_options_scan returned empty path")
            elif not os.path.exists(options_path):
                print(f"  ✗ OPTIONS  : SKIPPED — file not found at {options_path}")
                print(f"             (cwd={os.getcwd()})")
            else:
                attachments.append(options_path)
                size_kb = os.path.getsize(options_path) / 1024
                print(f"  ✓ OPTIONS  : {options_path} ({size_kb:.1f} KB)")
        else:
            print(f"  ○ OPTIONS  : disabled by EMAIL_ATTACH_OPTIONS=false")
        print(f"[Email] Total attachments queued: {len(attachments)}")
        # ──────────────────────────────────────────────────────────────────

        top10 = product_df.head(10)
        top10_lines = "\n".join(
            f"  {i+1:>2}. {r['ticker']:<8} {r.get('signal',''):<12} "
            f"OCC={r.get('occ_score','')}  Change={r.get('signal_change','')}  "
            f"Days={r.get('days_in_product','')}  ATR%={r.get('atr_pct','')}  "
            f"RelVol={r.get('rel_volume','')}"
            for i, (_, r) in enumerate(top10.iterrows())
        ) if not top10.empty else "  (none)"

        body_plain = (
            f"BrokeBull Stage-2 Engine v2.3  |  {today}\n"
            f"{'─'*60}\n"
            f"Universe: NASDAQ · Other Listed · S&P 500/400/600 · "
            f"Russell 1000/2000/3000 · DJ Wilshire 5000 · MSCI World/EAFE · Nikkei 225\n"
            f"Options: SPY/QQQ/IWM + Mag 7 (calls & puts, OTM/ATM, {OPT_MIN_DTE}-{OPT_MAX_DTE} DTE)\n"
            f"{'─'*60}\n"
            f"TECH:    {len(tech_df)} tickers passed all gates\n"
            f"QUALITY: {len(quality_df)} tickers passed valuation filters\n"
            f"PRODUCT: {len(product_df)} top-tier picks\n\n"
            f"TOP 10 PRODUCT PICKS:\n{top10_lines}\n\n"
            f"Config: {cfg.STAGE2_WEEKLY_MA_WEEKS}W gate | "
            f"{cfg.DAILY_SMA_DAYS}d SMA (req={cfg.REQUIRE_DAILY_200SMA_CONFIRM}) | "
            f"Floor={cfg.PRODUCT_MIN_TIER}\n"
            f"See attached CSVs for full results.\n"
        )

        body_html = _build_html_body(today, tech_df, quality_df, product_df, cfg,
                                     watchlist_df=watchlist_df, sector_data=sector_data)

        subject = (
            f"🐂 BrokeBull Stage2 — {today} — "
            f"TECH:{len(tech_df)} | QUALITY:{len(quality_df)} | PRODUCT:{len(product_df)}"
        )

        print(f"Sending email to: {os.getenv('EMAIL_TO', cfg.EMAIL_TO)}")
        print(f"Attachments queued: {[os.path.basename(a) for a in attachments]}")
        ok, msg = send_email_with_attachments(cfg, subject, body_plain, attachments, body_html)
        print(f"Email: {'✓' if ok else '✗'} {msg}")
    else:
        print("Email skipped — credentials not configured. Set SMTP_USER / SMTP_PASS / EMAIL_TO.")

    return tech_df, quality_df, product_df, watchlist_df


# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    cfg = CFG()
    tech_df, quality_df, product_df, watchlist_df = run_stage2_pipeline(cfg)

    print("\n── TECH (top 10) ──────────────────────────────")
    print(tech_df.head(10).to_string(index=False))
    print("\n── QUALITY (top 10) ───────────────────────────")
    print(quality_df.head(10).to_string(index=False))
    print("\n── PRODUCT (top 10) ───────────────────────────")
    print(product_df.head(10).to_string(index=False))
    print("\n── WATCHLIST (top 10) ─────────────────────────")
    print(watchlist_df.head(10).to_string(index=False) if not watchlist_df.empty else "(empty)")

[Cache Status]
      date period  tickers                 fetched_at  bad_symbols  size_mb
2026-04-27    10y     5956 2026-04-27T14:34:03.969925         3451   303.88

[Universe] Fetching NASDAQ Listed + Other Listed …
[Universe] NASDAQ/Other: 5823 symbols
[Universe] Fetching extended index constituents (S&P/Russell/Wilshire/MSCI/Nikkei) …
[Universe/sp_smallcap_600] 608 equity tickers loaded.
[Universe/sp_midcap_400] 406 equity tickers loaded.
[Universe/russell_2000] 1929 equity tickers loaded.
[Universe/sp500] 505 equity tickers loaded.
[Universe/russell_1000] 1005 equity tickers loaded.
[Universe/dj_wilshire_5000] 2504 equity tickers loaded.
[Universe/msci_eafe] 412 equity tickers loaded.
[Universe/russell_3000] 2579 equity tickers loaded.
[Universe/nikkei_225] 2 equity tickers loaded.
[Universe/msci_world_acwi] 1000 equity tickers loaded.
[Universe]   sp500: 505 symbols after clean
[Universe]   sp_midcap_400: 406 symbols after clean
[Universe]   sp_smallcap_600: 608 symbols after cl

/tmp/ipykernel_704/3421388530.py:1867: RuntimeWarning: Mean of empty slice
  prof_s = np.nanmean([roe_s, netm_s])
/tmp/ipykernel_704/3421388530.py:1872: RuntimeWarning: Mean of empty slice
  val_s  = np.nanmean([pe_s, peg_s, ev_s])


Wrote TECH: ./output/stage2_TECH_2026-04-27.csv (rows=321)

[Funnel] Failure breakdown of 321 TECH passes:
  ── Absolute valuation checks ──
    val_eps_pos                  failed:   87 ( 27.1%)
    val_roe                      failed:  108 ( 33.6%)
    val_net_margin               failed:   93 ( 29.0%)
    val_pe                       failed:   46 ( 14.3%)
    val_peg                      failed:   56 ( 17.4%)
    val_pb                       failed:   48 ( 15.0%)
    val_ps                       failed:   42 ( 13.1%)
    val_ev_ebitda                failed:   99 ( 30.8%)
    val_fcf_yield                failed:   75 ( 23.4%)
    val_debt_to_equity           failed:   35 ( 10.9%)
  ── Sector-relative checks ──
    rel_pe_ok                    failed:   50 ( 15.6%)
    rel_ev_ebitda_ok             failed:   80 ( 24.9%)
    rel_peg_ok                   failed:   37 ( 11.5%)
    rel_ps_ok                    failed:   68 ( 21.2%)
    rel_pb_ok                    failed:   71 ( 22.1%)
   